<sub>**Codeverra** · Concurrency Track · Workbook 1</sub>

# The Concurrency Workbook

### Six Case Studies in Threads, Processes, and asyncio

You have already met the vocabulary of concurrent programming: processes and threads, the Global Interpreter Lock, coroutines and the event loop, locks, semaphores, queues, and executors. Vocabulary is the easy part. The difficulty in real work is deciding *which* of these tools a given problem calls for, recognising the failure that appears when the wrong one is chosen, and being able to defend the decision under questioning.

This workbook exists to convert that vocabulary into judgment. It contains six case studies. Each one begins with a small piece of working code that has a defect, a bottleneck, or a design weakness. You read the code, reason about what will happen when several things run at once, and then refactor it. Every claim made in the text is backed by a measurement you run yourself, because concurrency is the one area of programming where intuition is least reliable and a stopwatch is most persuasive.

**How to work through it.** Run every cell in order. Read the analysis *before* running the cell that follows it, and predict the output. Where a case study asks you to guess a number, write the guess down before executing. The gap between your prediction and the measurement is the actual lesson.

**What you need.** Python 3.9 or newer and nothing else. There are no third-party dependencies. All network calls are simulated so the notebook runs offline and produces stable timings; the simulation uses the same blocking and yielding behaviour as a real socket, which is explained in the setup section.

**A note on your machine.** Several measurements depend on how many CPU cores you have. The notebook prints your core count during setup. If you are running inside a single-core container, the multiprocessing sections will show no speedup, and the text explains exactly why that is the correct result rather than a broken example.

---

## Contents

| # | Case study | Concurrency concepts it teaches |
|---|------------|--------------------------------|
| 0 | [Orientation](#0) | Concurrency vs parallelism, I/O-bound vs CPU-bound, the GIL |
| 1 | [The Thumbnail Service](#cs1) | Choosing a model, `ThreadPoolExecutor`, `ProcessPoolExecutor`, `asyncio.gather`, hybrid pipelines |
| 2 | [The Flash Sale That Oversold](#cs2) | Race conditions, `Lock`, lock granularity, deadlock and prevention, `RLock`, `asyncio.Lock` |
| 3 | [The Payment Gateway With a Ceiling](#cs3) | `Semaphore`, `BoundedSemaphore`, concurrency limits vs rate limits, token buckets, `asyncio.Semaphore` |
| 4 | [The Log Ingestion Pipeline](#cs4) | `queue.Queue`, producer and consumer, backpressure, sentinels, graceful shutdown, `asyncio.Queue` |
| 5 | [The Cache That Stampeded](#cs5) | Single flight, `Event`, `Condition`, predicate loops, `Barrier`, `threading.local` |
| 6 | [The Dashboard That Missed Its Deadline](#cs6) | `Future`, `as_completed`, exception propagation, timeouts, cancellation, `TaskGroup`, `asyncio.to_thread` |
|   | [Decision framework](#framework) | Choosing a model under pressure |
|   | [Summary and key takeaways](#summary) | |
|   | [Practice problems with solutions](#practice) | Eight problems, interview style |

---

<a id="0"></a>
## 0. Orientation

Before the first case study, three distinctions need to be firmly in place, because every decision in this workbook reduces to one of them.

### Concurrency and parallelism

**Concurrency** is a property of a program's structure: the work has been divided into units that can be started, paused, and resumed independently, so that progress on one unit does not require another to finish. **Parallelism** is a property of execution: two or more units genuinely run at the same instant on different processors.

A single-core machine can run a highly concurrent program with no parallelism at all, interleaving many tasks so quickly that all of them appear to advance together. A program can also be parallel without being interesting concurrently, as when a numerical library splits one matrix multiplication across eight cores. The two properties are related but independent, and confusing them leads to a specific and common mistake: expecting a concurrency tool to speed up work that needs parallelism.

### I/O-bound and CPU-bound work

A task is **I/O-bound** when its elapsed time is dominated by waiting for something outside the CPU: a network response, a disk read, a database query, a message from another process. During that wait the CPU has nothing to do on behalf of this task.

A task is **CPU-bound** when its elapsed time is dominated by computation: parsing, encoding, compressing, resizing, hashing in a loop, scoring a model. The CPU is busy for the entire duration.

The distinction matters because it determines what "going faster" even means. Overlapping ten waits is free, since waiting consumes no processor. Overlapping ten computations is not free, because it requires ten processors, or at least more than one.

### The Global Interpreter Lock

CPython protects its internal state with a single lock, the **Global Interpreter Lock (GIL)**. Any thread that wants to execute Python bytecode must hold it, so only one thread executes Python bytecode at a time within one process, no matter how many cores the machine has.

Two consequences follow, and both are central to this workbook.

The first consequence is that threads do not give you parallel computation in CPython. Four threads running four pure-Python loops finish in roughly the time one thread would take, plus overhead. Only separate processes, each with its own interpreter and its own GIL, achieve parallel computation.

The second consequence is that the GIL is released whenever a thread is about to wait. Blocking calls in the standard library and in C extensions release the GIL before they block and reacquire it afterwards, which means threads *do* give you overlapping I/O. This is why threading remains the right answer for I/O-bound work despite the GIL.

A third point is frequently misunderstood and causes real production bugs: the GIL makes a single bytecode instruction atomic, and nothing larger. A Python statement usually compiles to several bytecode instructions, and the interpreter may switch threads between any two of them. Case Study 2 measures exactly what that costs.

### The three models at a glance

| | Threading | Multiprocessing | asyncio |
|---|---|---|---|
| Unit of concurrency | OS thread | OS process | Coroutine on one thread |
| Scheduling | Preemptive, by the OS | Preemptive, by the OS | Cooperative, at `await` points |
| Memory | Shared | Isolated, copied or piped | Shared |
| Parallel CPU work | No, blocked by the GIL | Yes | No |
| Overlapping I/O | Yes | Yes, wastefully | Yes |
| Typical creation cost | Roughly 50 to 100 microseconds, about 8 MB of virtual address space per stack | Milliseconds, plus interpreter startup on spawn | Microseconds, a few kilobytes per task |
| Practical ceiling | Hundreds to a few thousand | Number of cores | Tens of thousands |
| Where correctness bites | Locks around shared state | Serialising arguments and results | Never blocking the loop |
| Standard entry point | `ThreadPoolExecutor` | `ProcessPoolExecutor` | `asyncio.run` |

Keep this table in view. Every case study that follows is an argument for one row of it.

---

### Setup

The next cell writes a small module called `clab.py` next to this notebook. Everything the case studies share lives there: a timing helper, a simulated download, and a genuinely CPU-bound image filter.

Placing these functions in a real module file rather than in a notebook cell is a practical necessity, not a stylistic choice. On macOS and Windows, `multiprocessing` starts worker processes with the **spawn** method, which launches a fresh interpreter and asks it to *import* the function it is supposed to run. A function defined in a notebook cell cannot be imported by a fresh interpreter, so process pools fail with a pickling error. A function defined in an importable module works on every platform. This is one of the most common reasons a multiprocessing example works for the person who wrote it and fails for everyone else.

Two design decisions in the module deserve a word.

**The simulated download.** `fetch_image` calls `time.sleep`. That is a faithful stand-in for a network request because `time.sleep` releases the GIL for the duration of the wait, exactly as a real socket read does. A thread sitting in `time.sleep` does not prevent other threads from running Python code, so every timing comparison in this workbook behaves the way it would against a real HTTP server, without the variance of an actual network.

**The simulated processing.** `make_thumbnail` performs a 3x3 box blur written in pure Python. Pure Python arithmetic holds the GIL for its entire duration, which makes it a faithful stand-in for any real CPU-bound step. Note that a production version using Pillow or NumPy would behave differently, because those libraries release the GIL inside their C routines. That difference is discussed at the end of Case Study 1.

In [ ]:
%%writefile clab.py
"""
clab.py : Concurrency Lab toolkit for the Codeverra Concurrency Workbook.

Everything in this file is deliberately dependency-free so the workbook runs
on a plain CPython 3.9+ installation. It lives in a real module file (rather
than inside a notebook cell) for one important reason: on macOS and Windows,
multiprocessing starts worker processes with the "spawn" method, and a spawned
worker can only run functions it can *import*. Functions defined in a notebook
cell are not importable, so process pools would fail. Functions defined here
are importable everywhere.
"""

from __future__ import annotations

import asyncio
import threading
import time
import zlib
from contextlib import contextmanager

# --------------------------------------------------------------------------
# Timing
# --------------------------------------------------------------------------

@contextmanager
def timed(label: str):
    """Print how long the enclosed block took. Used for every measurement."""
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"{label:<44} {elapsed:6.2f}s")


# --------------------------------------------------------------------------
# Simulated I/O-bound work: "downloading" an image
# --------------------------------------------------------------------------

NETWORK_LATENCY = 0.30  # seconds per simulated request


def synthetic_image(seed: int, size: int = 96) -> list:
    """
    Produce a deterministic size x size greyscale image as a flat list of ints.
    A tiny linear congruential generator keeps this fast and reproducible.
    """
    pixels = []
    state = seed or 1
    for _ in range(size * size):
        state = (1103515245 * state + 12345) & 0x7FFFFFFF
        pixels.append(state % 256)
    return pixels


def fetch_image(url: str, latency: float = NETWORK_LATENCY) -> list:
    """
    A blocking 'download'.

    time.sleep() releases the GIL while it waits, which is exactly what a real
    socket read does. The timing behaviour of this function under threads,
    processes and asyncio therefore matches a real HTTP call closely enough
    for every lesson in this workbook.
    """
    time.sleep(latency)
    return synthetic_image(zlib.crc32(url.encode()))


async def fetch_image_async(url: str, latency: float = NETWORK_LATENCY) -> list:
    """The asyncio-native counterpart of fetch_image."""
    await asyncio.sleep(latency)
    return synthetic_image(zlib.crc32(url.encode()))


# --------------------------------------------------------------------------
# CPU-bound work: turning an image into a thumbnail
# --------------------------------------------------------------------------

def blur(pixels: list, size: int = 96, passes: int = 6) -> list:
    """
    A 3x3 box blur written in pure Python.

    Pure Python arithmetic holds the GIL for its entire duration, which makes
    this a faithful stand-in for any CPU-bound step: image filtering, parsing,
    compression, hashing in a loop, model scoring.
    """
    current = pixels
    for _ in range(passes):
        out = [0] * (size * size)
        for y in range(size):
            for x in range(size):
                total = 0
                count = 0
                for dy in (-1, 0, 1):
                    ny = y + dy
                    if ny < 0 or ny >= size:
                        continue
                    row = ny * size
                    for dx in (-1, 0, 1):
                        nx = x + dx
                        if 0 <= nx < size:
                            total += current[row + nx]
                            count += 1
                out[y * size + x] = total // count
        current = out
    return current


def make_thumbnail(pixels: list, size: int = 96, factor: int = 4) -> list:
    """Blur, then downsample by averaging each factor x factor block."""
    smoothed = blur(pixels, size=size)
    small = size // factor
    out = []
    for y in range(small):
        for x in range(small):
            total = 0
            for dy in range(factor):
                row = (y * factor + dy) * size
                for dx in range(factor):
                    total += smoothed[row + x * factor + dx]
            out.append(total // (factor * factor))
    return out


def download_and_thumbnail(url: str) -> list:
    """One complete unit of work: I/O-bound fetch followed by CPU-bound processing."""
    return make_thumbnail(fetch_image(url))


# --------------------------------------------------------------------------
# Running asyncio code from a notebook
# --------------------------------------------------------------------------

def run_async(coro):
    """
    Run a coroutine to completion and return its result.

    asyncio.run() creates a fresh event loop, runs the coroutine, and closes
    the loop. It refuses to start if a loop is already running in the current
    thread, which is precisely the situation inside a Jupyter kernel. When a
    loop is already running we execute the coroutine on a dedicated thread
    that owns its own private loop, so the same call works in a notebook, in
    a plain script, and inside another framework's loop.
    """
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    box = {}

    def runner():
        try:
            box["value"] = asyncio.run(coro)
        except BaseException as exc:  # noqa: BLE001 - re-raised on the caller's thread
            box["error"] = exc

    thread = threading.Thread(target=runner)
    thread.start()
    thread.join()
    if "error" in box:
        raise box["error"]
    return box["value"]


IMAGE_URLS = [f"https://images.example.com/photo_{i:02d}.jpg" for i in range(8)]


In [ ]:
import os
import sys
import platform

import clab

print(f"Python           {sys.version.split()[0]}")
print(f"Platform         {platform.system()} ({platform.machine()})")
print(f"Logical CPUs     {os.cpu_count()}")
print(f"Simulated latency per download: {clab.NETWORK_LATENCY:.2f}s")

if os.cpu_count() == 1:
    print("\nNOTE: this machine reports a single logical CPU. Multiprocessing")
    print("sections will show no speedup. Read the explanation there; that")
    print("outcome is the correct one for a single core, not a broken example.")

---

<a id="cs1"></a>
# Case Study 1: The Thumbnail Service

## The situation

A media team runs a nightly job that pulls photographs from a partner's content delivery network and generates thumbnails for the internal catalogue. The job worked well when the partner supplied a few dozen images a night. The partner now supplies several thousand, and the job no longer finishes before the morning publishing deadline.

Here is the job, reduced to its essentials. Read it before running it, and form an opinion about where the time goes.

In [ ]:
from clab import IMAGE_URLS, fetch_image, make_thumbnail, timed


def build_thumbnails_sequential(urls):
    """Download each image, then reduce it to a thumbnail. One at a time."""
    thumbnails = []
    for url in urls:
        raw = fetch_image(url)                 # waits on the network
        thumbnails.append(make_thumbnail(raw))  # computes on the CPU
    return thumbnails


print(f"{len(IMAGE_URLS)} images to process\n")

with timed("v1  sequential"):
    thumbnails = build_thumbnails_sequential(IMAGE_URLS)

print(f"\nproduced {len(thumbnails)} thumbnails of {len(thumbnails[0])} pixels each")

## Step 1: Find out where the time actually goes

The instinct at this point is to reach for threads. Resist it for one more cell. A concurrency decision made before measuring the split between waiting and computing is a guess, and the wrong guess produces code that is more complicated and no faster.

The loop does two different kinds of work per image. Time them separately.

In [ ]:
# Measure the two halves of the pipeline independently.
raw_images = []

with timed("downloading only"):
    raw_images = [fetch_image(url) for url in IMAGE_URLS]

with timed("processing only"):
    _ = [make_thumbnail(raw) for raw in raw_images]

The download phase dominates. With a simulated latency of 0.30 seconds and eight images, roughly 2.4 seconds of the job is spent with the CPU completely idle, waiting for bytes to arrive. The processing phase is real work, but it is a much smaller share of the total.

That measurement settles the first decision. The largest available win comes from overlapping the waiting, and overlapping waiting is what threads and asyncio both do well.

Two properties of this workload make overlapping safe and easy, and it is worth naming them because their absence is what makes other problems hard:

- The tasks are **independent**. Thumbnail number three does not need anything produced by thumbnail number two.
- The tasks share **no mutable state**. Each one reads a URL and returns a new list. Nothing is written to a variable that another task can see.

Work with these two properties is called *embarrassingly parallel*, and it is the easiest kind of concurrency there is. Case Studies 2 through 5 all deal with what happens when one of the two properties fails.

## Step 2: Overlap the waiting with threads

A **thread** is an independently schedulable sequence of instructions inside a process. Threads in one process share the same memory, which makes communication between them free and makes uncoordinated writes dangerous. Since our tasks share nothing, only the first property matters here.

Rather than creating threads by hand, use `concurrent.futures.ThreadPoolExecutor`. An **executor** is a managed pool of workers with a queue in front of it. You submit callables; it runs them on the workers and hands back results. The pool is created once and reused, which avoids paying thread creation cost per task, and the `with` block guarantees that every worker is shut down cleanly even if the body raises.

`pool.map` is the closest analogue to the built-in `map`: it applies one function to every item and yields results **in input order**, regardless of the order in which they actually finished.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from clab import download_and_thumbnail  # fetch_image followed by make_thumbnail

with timed("v2  ThreadPoolExecutor(8), whole pipeline"):
    with ThreadPoolExecutor(max_workers=8) as pool:
        thumbnails = list(pool.map(download_and_thumbnail, IMAGE_URLS))

print(f"\nproduced {len(thumbnails)} thumbnails")

The job now takes roughly one download latency plus the processing time, instead of eight download latencies plus the processing time.

The mechanism is worth stating precisely, because it is the single most useful thing to understand about threads in Python. When a worker thread calls `time.sleep` (standing in for a socket read), the interpreter releases the GIL before the wait begins. Another worker thread immediately acquires the GIL and runs until it, too, reaches a blocking call. Eight threads therefore sit in their waits at the same time, and the eight waits overlap almost perfectly. The GIL is not an obstacle here, because none of the waiting threads wants to execute bytecode while it waits.

The processing phase behaves entirely differently, and the next cell proves it.

## Step 3: Watch threads fail at the CPU-bound half

The blur runs entirely in Python bytecode, so a thread performing it holds the GIL for the whole computation. Adding threads therefore adds scheduling overhead and no throughput. Predict the result of the next cell before running it.

In [ ]:
from clab import blur

with timed("processing 8 images, 1 thread"):
    _ = [make_thumbnail(raw) for raw in raw_images]

with timed("processing 8 images, 4 threads"):
    with ThreadPoolExecutor(max_workers=4) as pool:
        _ = list(pool.map(make_thumbnail, raw_images))

The two timings are close, and the threaded version is often slightly slower. Four threads contending for one GIL do not compute four times faster; they take turns computing at the speed of one, and pay for the turn-taking.

This is the clearest way to internalise the rule: **threads overlap waiting, not computing.** Whenever someone reports that adding threads made their Python program slower, this is nearly always why.

## Step 4: Parallelise the CPU-bound half with processes

To compute in parallel, the work must escape the GIL, and the only way to escape a per-interpreter lock is to use more than one interpreter. A **process** is an independently scheduled program with its own private memory space and, in CPython, its own GIL. Two processes running Python code genuinely execute at the same time on two cores.

`ProcessPoolExecutor` presents the same interface as `ThreadPoolExecutor`, which makes the swap a one-word change in the source. The cost model underneath is completely different, and the difference has three parts.

**Startup.** Each worker is a new operating system process. On Linux the default start method is `fork`, which is fast. On macOS and Windows the default is `spawn`, which launches a fresh interpreter and re-imports your modules, costing tens to hundreds of milliseconds per worker.

**Serialisation.** Arguments and results cross a process boundary, so they are converted to bytes with `pickle`, written to a pipe, and reconstructed on the other side. Anything that cannot be pickled, such as an open socket, a database connection, a lambda, or a locally defined function, cannot be sent. Large arguments cost real time and memory to move.

**No shared memory.** A worker cannot see or modify the parent's variables. Communication must be explicit.

The rule of thumb that follows: a task is worth sending to a process pool when the computation it performs clearly exceeds the cost of shipping its inputs and outputs across the boundary. Sixteen images with a few tens of milliseconds of work each sit comfortably above that line; sixteen additions would sit far below it.

In [ ]:
import os
from concurrent.futures import ProcessPoolExecutor

# Enough work that the computation clearly outweighs the cost of moving data
# between processes.
cpu_batch = raw_images * 2   # 16 images

with timed("processing 16 images, sequential"):
    _ = [make_thumbnail(raw) for raw in cpu_batch]

with timed(f"processing 16 images, ProcessPoolExecutor({os.cpu_count()})"):
    with ProcessPoolExecutor() as pool:
        # chunksize batches several images into one message, which amortises
        # the per-task pickling and dispatch overhead across the batch.
        _ = list(pool.map(make_thumbnail, cpu_batch, chunksize=2))

On a machine with four or more cores this shows a clear speedup, bounded above by the core count and reduced by the serialisation cost. On a single-core machine the two timings are the same or the pool is slower, which is exactly right: there is no second processor for the second process to run on, so all you have bought is overhead.

Two practical notes follow from the measurement.

`chunksize` matters for short tasks. Without it, `map` dispatches one task per message, and for tasks lasting a few milliseconds the messaging dominates. Grouping several tasks per message amortises that cost. For long tasks, leave it at the default so that work stays evenly balanced across workers.

The upper bound on speedup is the number of physical cores, not the number of workers. Creating sixty-four processes on an eight-core machine produces sixty-four processes competing for eight cores, plus sixty-four times the memory. `ProcessPoolExecutor()` with no arguments defaults to `os.cpu_count()`, which is the right starting point for CPU-bound work.

## Step 5: The same overlap with asyncio

Threads are not the only way to overlap waiting. **asyncio** achieves the same overlap on a single thread by making the waiting explicit in the source code.

A **coroutine** is a function declared with `async def` whose execution can be suspended at marked points and resumed later. Calling it does not run the body; it returns a coroutine object describing work to be done. The **event loop** is the scheduler that runs those objects. It keeps a queue of coroutines ready to run and a set of coroutines waiting on I/O, runs a ready one until it suspends, and when nothing is ready asks the operating system to wake it as soon as any awaited operation completes.

The `await` keyword marks a suspension point. `await asyncio.sleep(0.3)` tells the loop that this coroutine has nothing to do for 0.3 seconds and that the loop should run something else in the meantime. This is **cooperative** scheduling: a coroutine keeps the thread until it chooses to yield, in contrast to threads, which the operating system can preempt at any instruction.

`asyncio.gather` schedules several coroutines concurrently and waits for all of them, returning results in the order the coroutines were passed in.

One practical wrinkle appears immediately in a notebook. `asyncio.run` creates a new event loop and refuses to start if one is already running in the current thread, which is exactly the situation inside a Jupyter kernel. The helper `clab.run_async` handles both cases: it calls `asyncio.run` when no loop is running, and otherwise executes the coroutine on a dedicated thread that owns its own private loop. Outside a notebook, `asyncio.run(main())` is the standard entry point and is what you should write.

In [ ]:
import asyncio

from clab import fetch_image_async, run_async


async def download_all(urls):
    """Schedule every download at once and wait for all of them."""
    return await asyncio.gather(*(fetch_image_async(url) for url in urls))


with timed("v3  asyncio.gather, 8 downloads"):
    raws = run_async(download_all(IMAGE_URLS))

print(f"\ndownloaded {len(raws)} images on a single thread")

The elapsed time matches the threaded download almost exactly, which raises the obvious question: if both approaches overlap the same waiting, why does asyncio exist?

## Step 6: Why asyncio wins at scale

The answer is cost per unit of concurrency, and it only becomes visible when the number of simultaneous operations grows.

Every thread receives an operating system stack. On Linux the default reservation is eight megabytes of virtual address space per thread, and each thread is an object the kernel scheduler must track and switch between. Ten thousand threads is impractical on most systems. Every coroutine, by contrast, is an ordinary Python object holding a few kilobytes of state, scheduled by a loop in user space. Ten thousand coroutines is unremarkable.

The next cell measures pure creation and scheduling overhead, with no work at all in each unit, so that the structural cost is isolated.

In [ ]:
import threading
import time

N = 1000


def nothing():
    pass


async def nothing_async():
    pass


start = time.perf_counter()
threads = [threading.Thread(target=nothing) for _ in range(N)]
for t in threads:
    t.start()
for t in threads:
    t.join()
thread_cost = time.perf_counter() - start


async def create_many_tasks():
    tasks = [asyncio.create_task(nothing_async()) for _ in range(N)]
    await asyncio.gather(*tasks)


start = time.perf_counter()
run_async(create_many_tasks())
task_cost = time.perf_counter() - start

print(f"{N} threads   created, run and joined : {thread_cost*1000:8.1f} ms")
print(f"{N} coroutine tasks created and run   : {task_cost*1000:8.1f} ms")
print(f"\nratio: threads cost about {thread_cost/task_cost:.0f}x more per unit")

The ratio is typically a factor of ten or more, and it widens as N grows because thread creation scales worse than task creation. The memory ratio is larger still: megabytes of stack reservation per thread against a few kilobytes per task.

That is the whole argument for asyncio. For a handful of concurrent operations, threads are simpler and perform identically, and `ThreadPoolExecutor` around an existing blocking library is often the pragmatic answer. For thousands of simultaneous connections, threads stop being viable and asyncio is the only option in standard Python.

There is a second, less quantitative argument. Because a coroutine can only be suspended at an `await`, you can read a coroutine and see every point at which another task might interleave. In threaded code, a switch can occur between any two bytecode instructions, so every shared variable is a potential race. Cooperative scheduling makes a large class of race conditions structurally impossible, which is why Case Study 2 has far less to say about locks in the asyncio section than in the threading section.

The cost of that guarantee is the discipline it demands: a coroutine that computes for 200 milliseconds without awaiting freezes every other coroutine on the loop for 200 milliseconds. Case Study 6 measures this failure directly.

## Step 7: The pipeline that uses the right tool for each half

The original job has an I/O-bound phase and a CPU-bound phase. Nothing requires both to use the same model.

In [ ]:
def build_thumbnails_hybrid(urls):
    """asyncio for the waiting, processes for the computing."""
    raws = run_async(download_all(urls))          # overlapped I/O, one thread
    with ProcessPoolExecutor() as pool:           # parallel CPU, many cores
        return list(pool.map(make_thumbnail, raws, chunksize=2))


with timed("v4  asyncio downloads + process pool processing"):
    thumbnails = build_thumbnails_hybrid(IMAGE_URLS)

print(f"\nproduced {len(thumbnails)} thumbnails")

This structure, an async front end feeding a process pool, is the shape of a great many production services: an asyncio web server accepts requests and performs all its network calls on the loop, and hands genuinely CPU-heavy work to a pool of worker processes so the loop stays responsive. Case Study 6 shows how to make that handoff from *inside* a coroutine using `loop.run_in_executor`, so that the front end never blocks at all.

## Results and interpretation

| Version | Model | What it overlaps | Approximate cost |
|---|---|---|---|
| v1 sequential | none | nothing | 8 latencies + 8 computations |
| v2 thread pool | threading | all the waiting | 1 latency + 8 computations |
| v3 asyncio | asyncio | all the waiting | 1 latency, downloads only |
| CPU with threads | threading | nothing useful | 8 computations, plus overhead |
| CPU with processes | multiprocessing | the computing | 8 computations divided by cores, plus transfer |
| v4 hybrid | both | waiting and computing | 1 latency + 8 computations divided by cores |

Three conclusions carry forward to everything that follows.

**Measure the split before choosing a model.** The correct choice is fully determined by how much of the elapsed time is waiting and how much is computing, and that ratio is cheap to measure and impossible to guess reliably.

**Threads and asyncio solve the same problem at different scales.** Both overlap waiting. Threads cost more per unit and require no rewrite of blocking library calls. Coroutines cost far less per unit and require an async-native library for every I/O call in the path.

**Processes are the only route to parallel Python computation, and they are not free.** Startup, pickling, and the absence of shared memory all have to be paid for. The work sent across the boundary must be large enough to justify the crossing.

## Interview follow-ups

*The team replaces the pure-Python blur with Pillow. Does the analysis change?*

Yes, in an important way. Pillow performs its work in C and releases the GIL while doing so, so a thread running a Pillow resize does not block other Python threads. Image processing with Pillow, array mathematics with NumPy, compression with `zlib`, and hashing with `hashlib` over large buffers all behave more like I/O than like pure-Python computation from the GIL's point of view, and thread pools can parallelise them effectively. The general principle is that the GIL blocks *Python bytecode*, so the relevant question is always how much of the hot path executes as bytecode rather than inside a C routine.

*Would you use processes for the downloads too, since they overlap waiting as well?*

It would work and it would be wasteful. Each process consumes megabytes of memory and milliseconds of startup to do something that costs a coroutine a few kilobytes: sit still and wait. Processes buy parallel computation, and you should pay for them only when that is what you need.

*Downloads now come from an API that permits at most five concurrent connections. What changes?*

The pool size stops being a tuning parameter and becomes a correctness constraint. `ThreadPoolExecutor(max_workers=5)` enforces it when the pool is dedicated to that API, but as soon as several call sites share the same limit, the limit belongs to the resource rather than to any one pool, and the right tool is a semaphore. That is Case Study 3.

*How would you handle one URL that fails?*

`pool.map` raises the first exception when you iterate to the failed element and abandons the remaining results, which is rarely what a batch job wants. The alternative is `submit`, which returns a `Future` per task and lets you inspect each outcome independently. Case Study 6 covers exactly this.

---

<a id="cs2"></a>
# Case Study 2: The Flash Sale That Oversold

## The situation

An e-commerce team runs a flash sale. Fifty units of a phone go on sale at noon, and roughly two hundred customers press "buy" within the same second. The web server handles each request on its own thread, and every thread calls the same method on a shared `Inventory` object.

The sale ends with fifty units in stock and a number of confirmed orders that is comfortably larger than fifty. Support spends the afternoon cancelling orders.

Here is the inventory service. It is correct in the sense that a single-threaded reading of it is flawless.

In [ ]:
import threading
import time


class Inventory:
    """Tracks units in stock per SKU. Single-threaded, this is correct."""

    def __init__(self, stock):
        self._stock = dict(stock)

    def reserve(self, sku, quantity=1):
        available = self._stock[sku]          # 1. read the current level
        if available >= quantity:             # 2. check that it is sufficient
            time.sleep(0)                     # see the note below
            self._stock[sku] = available - quantity   # 3. write the new level
            return True
        return False

    def level(self, sku):
        return self._stock[sku]


def run_flash_sale(inventory, sku, buyers=200):
    """Simulate `buyers` customers each trying to reserve one unit at once."""
    confirmed = []

    def buy(customer_id):
        if inventory.reserve(sku, 1):
            # list.append compiles to a single bytecode instruction and is
            # therefore safe here without a lock. That is a CPython
            # implementation detail rather than a language guarantee, and it
            # is exactly the kind of assumption worth stating out loud.
            confirmed.append(customer_id)

    threads = [threading.Thread(target=buy, args=(i,)) for i in range(buyers)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return confirmed


inventory = Inventory({"phone-x": 50})
confirmed = run_flash_sale(inventory, "phone-x", buyers=200)

print(f"units originally in stock : 50")
print(f"orders confirmed          : {len(confirmed)}")
print(f"stock level afterwards    : {inventory.level('phone-x')}")
print(f"units oversold            : {max(0, len(confirmed) - 50)}")

## Step 1: Name the defect precisely

The output shows more confirmed orders than units. The bug is a **race condition**: a defect whose presence depends on the relative timing of two or more threads, when that timing is not controlled.

The specific shape here is called **check-then-act**. The method reads a value, makes a decision based on it, and then acts on that decision. Between the check and the act, the value can change, which makes the decision stale by the time it is used.

Trace two threads through the method with fifty units left:

| Time | Thread A | Thread B | Stock |
|---|---|---|---|
| 1 | reads `available = 50` | | 50 |
| 2 | | reads `available = 50` | 50 |
| 3 | `50 >= 1`, proceeds | | 50 |
| 4 | | `50 >= 1`, proceeds | 50 |
| 5 | writes `50 - 1 = 49` | | 49 |
| 6 | | writes `50 - 1 = 49` | 49 |

Two units were sold and one unit was deducted. The write performed by thread A was silently discarded by thread B, which is called a **lost update**.

### A word about the `time.sleep(0)`

`time.sleep(0)` adds no delay. It asks the interpreter to consider switching to another thread at that exact point, which turns an intermittent race into a reproducible one. Deleting it does not make the code correct. It makes the failure rare enough to survive code review, pass every test, and appear only under production traffic, which is strictly worse than a bug that fails loudly.

The window exists without the sleep because a Python statement is not atomic. `self._stock[sku] = available - quantity` compiles to several bytecode instructions, and CPython may switch threads between any two of them. This is the crucial correction to a widespread misreading of the GIL. The GIL guarantees that one bytecode instruction completes before another thread runs a bytecode instruction. It guarantees nothing at all about a statement, a method, or a sequence of statements.

Confirm that with the disassembler.

In [ ]:
import dis

# A single, innocuous-looking statement.
source = "stock['phone-x'] = available - quantity"
print(f"{source}\n")
dis.dis(compile(source, "<demo>", "exec"))
print("\nSeveral instructions. A thread switch may occur between any two of them.")

## Step 2: The fix, and what it is actually protecting

A **lock** (also called a mutex) is an object with two operations and one rule. `acquire()` claims it, `release()` gives it up, and at most one thread may hold it at any moment. A thread that calls `acquire()` on a held lock waits until the holder releases it.

Wrapping the entire check-then-act sequence in a lock makes that sequence **atomic** from any other thread's point of view: no other thread can observe or modify the stock level partway through.

Two details in the implementation below matter more than they appear to.

**Use `with lock:` rather than manual `acquire` and `release`.** The context manager releases the lock even if the body raises. A lock released only on the success path becomes a permanent deadlock the first time an exception is thrown while it is held.

**Use one lock per SKU rather than one lock for the whole inventory.** The invariant being protected is per SKU: "if this SKU has stock at the moment of checking, the checking thread is the one that claims it." Reserving a phone and reserving a laptop cannot interfere with each other, so forcing them to take turns costs throughput and buys nothing. This is the principle of **lock granularity**: the lock should be exactly as wide as the invariant, and no wider.

In [ ]:
class SafeInventory:
    """Same behaviour, made atomic with one lock per SKU."""

    def __init__(self, stock):
        self._stock = dict(stock)
        # A separate lock per SKU. Different SKUs never contend with each other.
        self._locks = {sku: threading.Lock() for sku in stock}

    def reserve(self, sku, quantity=1):
        with self._locks[sku]:            # nobody else can be inside this block
            available = self._stock[sku]
            if available >= quantity:
                time.sleep(0)             # still here, and now harmless
                self._stock[sku] = available - quantity
                return True
            return False

    def level(self, sku):
        with self._locks[sku]:
            return self._stock[sku]


inventory = SafeInventory({"phone-x": 50})
confirmed = run_flash_sale(inventory, "phone-x", buyers=200)

print(f"orders confirmed       : {len(confirmed)}")
print(f"stock level afterwards : {inventory.level('phone-x')}")
print(f"units oversold         : {max(0, len(confirmed) - 50)}")

Exactly fifty orders, every run. The `time.sleep(0)` is still there and no longer matters, which is the point: correct synchronisation makes the outcome independent of scheduling.

Note carefully what the lock protects. It does not protect the variable `self._stock`. It protects the *invariant* that spans the read, the comparison, and the write. Any code that participates in maintaining an invariant must run under the same lock, and identifying the full extent of the invariant is the entire skill. A common partial fix is to lock only the assignment, which changes nothing, because by that point two threads have already made their decisions from the same stale reading.

## Step 3: Measure what granularity costs

Claims about lock granularity are easy to make and easy to verify. The next cell compares one global lock against one lock per SKU, with four SKUs being sold concurrently and a small amount of simulated work inside the critical section.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

WRITE_COST = 0.002   # a persistence call inside the critical section
SKUS = ["phone-x", "tablet-y", "watch-z", "buds-w"]


class GlobalLockInventory:
    """One lock for the entire catalogue."""

    def __init__(self, stock):
        self._stock = dict(stock)
        self._lock = threading.Lock()

    def reserve(self, sku, quantity=1):
        with self._lock:
            if self._stock[sku] >= quantity:
                time.sleep(WRITE_COST)
                self._stock[sku] -= quantity
                return True
            return False


class PerSkuLockInventory:
    """One lock per SKU."""

    def __init__(self, stock):
        self._stock = dict(stock)
        self._locks = {sku: threading.Lock() for sku in stock}

    def reserve(self, sku, quantity=1):
        with self._locks[sku]:
            if self._stock[sku] >= quantity:
                time.sleep(WRITE_COST)
                self._stock[sku] -= quantity
                return True
            return False


def hammer(inventory, orders=200):
    work = [SKUS[i % len(SKUS)] for i in range(orders)]
    with ThreadPoolExecutor(max_workers=16) as pool:
        list(pool.map(inventory.reserve, work))


from clab import timed

start_stock = {sku: 500 for sku in SKUS}

with timed("200 reservations, one global lock"):
    hammer(GlobalLockInventory(start_stock))

with timed("200 reservations, one lock per SKU"):
    hammer(PerSkuLockInventory(start_stock))

The per-SKU version finishes in roughly a quarter of the time, because four SKUs proceed independently while the global version forces all two hundred reservations into a single queue.

The trade-off runs in the other direction as well. Fine-grained locks improve throughput and increase the number of ways to get things wrong: more locks to remember to take, more chances to take them in an inconsistent order, and more difficulty reasoning about invariants that span two objects. That last risk is the subject of the next step.

## Step 4: The deadlock

The warehouse team adds a transfer operation that moves stock from one location to another. The two locations must both be locked, because the invariant now spans both: total stock across the pair must not change.

The implementation below locks the source first and the destination second, which reads naturally and is a classic defect. Two concurrent transfers in opposite directions acquire the two locks in opposite orders, and each ends up holding the lock the other one needs.

The demonstration runs the threads as daemons with a join timeout so that the notebook does not hang.

In [ ]:
class Warehouse:
    def __init__(self, name, units):
        self.name = name
        self.units = units
        self.lock = threading.Lock()


def transfer_unsafe(source, destination, units, delay=0.05):
    """Locks source then destination. Two opposite transfers deadlock."""
    with source.lock:
        time.sleep(delay)                 # widens the window; the bug is structural
        with destination.lock:
            source.units -= units
            destination.units += units


mumbai = Warehouse("mumbai", 100)
chennai = Warehouse("chennai", 100)

t1 = threading.Thread(target=transfer_unsafe, args=(mumbai, chennai, 10), daemon=True)
t2 = threading.Thread(target=transfer_unsafe, args=(chennai, mumbai, 10), daemon=True)
t1.start()
t2.start()
t1.join(timeout=2)
t2.join(timeout=2)

if t1.is_alive() or t2.is_alive():
    print("DEADLOCK: both threads are still waiting after 2 seconds.")
    print("Thread 1 holds mumbai and wants chennai.")
    print("Thread 2 holds chennai and wants mumbai.")
    print("Neither will ever proceed. These threads are daemons, so the")
    print("notebook continues, but in a server they would be lost forever.")
else:
    print("Completed without deadlock this time. Re-run the cell; the")
    print("outcome depends on timing, which is what makes the bug dangerous.")

## Step 5: Prevent the deadlock rather than detect it

A deadlock requires four conditions to hold simultaneously, a formulation due to Coffman:

1. **Mutual exclusion.** The resources cannot be shared.
2. **Hold and wait.** A thread holding one resource requests another.
3. **No preemption.** A resource cannot be forcibly taken from its holder.
4. **Circular wait.** A cycle exists in the graph of who waits for whom.

Breaking any one condition prevents deadlock. In practice, the fourth is the one to attack, because the first three are usually inherent to the problem.

The standard technique is a **global lock ordering**: define a total order over all locks and require every thread to acquire them in that order. A cycle cannot form, because forming one would require some thread to acquire a lower-ordered lock while holding a higher-ordered one, which the rule forbids. Any stable, unique key works as the order. Object identity via `id()` is convenient when there is no natural key; a database primary key or an account number is preferable in a distributed setting because it stays stable across processes.

Two alternatives are worth knowing.

**Acquire with a timeout.** `lock.acquire(timeout=...)` returns `False` instead of waiting forever, so the thread can release what it holds and retry. This converts a permanent hang into a livelock risk and needs randomised backoff to be safe. It is a reasonable fallback when a global ordering is impossible, as when lock identities are not known in advance.

**Use one coarser lock.** A single lock covering both warehouses removes the possibility of a cycle entirely, at the cost of serialising all transfers. When transfers are rare, this is the simplest correct answer, and simplicity has real value in concurrent code.

In [ ]:
def transfer_safe(source, destination, units, delay=0.05):
    """
    Always acquire the two locks in a fixed global order.
    id() gives a stable, unique ordering within one process; a database key
    would be the right choice if these objects lived in different processes.
    """
    first, second = sorted((source, destination), key=id)
    with first.lock:
        time.sleep(delay)
        with second.lock:
            source.units -= units
            destination.units += units


mumbai = Warehouse("mumbai", 100)
chennai = Warehouse("chennai", 100)

threads = [
    threading.Thread(target=transfer_safe, args=(mumbai, chennai, 10)),
    threading.Thread(target=transfer_safe, args=(chennai, mumbai, 10)),
    threading.Thread(target=transfer_safe, args=(mumbai, chennai, 25)),
    threading.Thread(target=transfer_safe, args=(chennai, mumbai, 25)),
]
for t in threads:
    t.start()
for t in threads:
    t.join(timeout=5)

print(f"alive threads after join : {sum(t.is_alive() for t in threads)}")
print(f"mumbai  : {mumbai.units}")
print(f"chennai : {chennai.units}")
print(f"total   : {mumbai.units + chennai.units} (invariant: 200)")

## Step 6: When a lock has to be re-entrant

A different self-inflicted deadlock appears when a method that holds a lock calls another method that acquires the same lock. A plain `Lock` has no notion of ownership: it only knows whether it is held. The second acquisition therefore waits for a release that can only come from the thread that is doing the waiting.

`threading.RLock`, a **re-entrant lock**, records which thread holds it and how many times. The owning thread may acquire it repeatedly, and the lock becomes available to others only after a matching number of releases.

Use `RLock` when a class has public methods that legitimately call one another. Use `Lock` otherwise, since it is marginally faster and, more usefully, it fails loudly when the call structure is more tangled than you thought.

In [ ]:
class Account:
    def __init__(self, balance):
        self.balance = balance
        # With threading.Lock() here, transfer() would deadlock against itself.
        self._lock = threading.RLock()

    def withdraw(self, amount):
        with self._lock:
            if self.balance >= amount:
                self.balance -= amount
                return True
            return False

    def deposit(self, amount):
        with self._lock:
            self.balance += amount

    def transfer_to(self, other, amount):
        with self._lock:              # first acquisition
            if self.withdraw(amount):  # second acquisition, same thread
                other.deposit(amount)
                return True
            return False


a, b = Account(100), Account(0)
print("transfer succeeded :", a.transfer_to(b, 40))
print("balances           :", a.balance, b.balance)

## Step 7: The same bug in asyncio, and why it is rarer

Coroutines on one event loop are scheduled cooperatively. A coroutine runs until it reaches an `await` and cannot be interrupted before that point. Any read, decision, and write that contains no `await` is therefore atomic with respect to other coroutines on the same loop, and needs no lock.

That guarantee ends at the first `await` inside a critical section. Suspending publishes the intermediate state to every other coroutine, which is precisely the window that a lock exists to close. Since realistic code awaits a database or an HTTP call in the middle of business logic, the check-then-act race returns in full.

The advantage over threads is that the danger is visible. Every point at which another task can interleave is spelled out in the source with the word `await`, so a reviewer can find the races by reading rather than by reasoning about arbitrary preemption.

In [ ]:
import asyncio

from clab import run_async


class AsyncInventory:
    """No lock. Safe only if the critical section contains no await."""

    def __init__(self, units):
        self.units = units

    async def reserve(self, quantity=1):
        available = self.units
        if available >= quantity:
            await asyncio.sleep(0)    # e.g. an await on the database
            self.units = available - quantity
            return True
        return False


class LockedAsyncInventory:
    def __init__(self, units):
        self.units = units
        self._lock = asyncio.Lock()   # note: an asyncio lock, not a threading lock

    async def reserve(self, quantity=1):
        async with self._lock:
            available = self.units
            if available >= quantity:
                await asyncio.sleep(0)
                self.units = available - quantity
                return True
            return False


async def flash_sale(inventory, buyers=200):
    results = await asyncio.gather(*(inventory.reserve() for _ in range(buyers)))
    return sum(results)


print("without asyncio.Lock:")
sold = run_async(flash_sale(AsyncInventory(50)))
print(f"  confirmed {sold} orders against 50 units\n")

print("with asyncio.Lock:")
sold = run_async(flash_sale(LockedAsyncInventory(50)))
print(f"  confirmed {sold} orders against 50 units")

`asyncio.Lock` and `threading.Lock` are not interchangeable. A `threading.Lock` blocks the whole thread, which on an event loop means blocking every coroutine including the one that would eventually release it, producing a deadlock of the entire application. An `asyncio.Lock` suspends only the waiting coroutine and lets the loop run others. The rule is that async code uses async primitives, and any `threading` primitive appearing in a coroutine is worth a second look. Case Study 6 covers the legitimate exception, where genuinely blocking code is pushed onto a worker thread.

## Takeaways

- A race condition is a defect whose presence depends on timing you do not control. Check-then-act is its most common shape, and lost updates are its most common symptom.
- The GIL makes one bytecode instruction atomic and nothing larger. Statements, methods and sequences of statements all need explicit synchronisation.
- A lock protects an invariant, not a variable. Every operation that participates in the invariant belongs inside the same lock.
- Lock granularity trades throughput against complexity. Match the lock to the invariant: per SKU, per account, per room, per key.
- Deadlock needs a cycle in the wait graph. A global lock ordering removes the possibility; timeouts and coarser locks are the fallbacks.
- Use `RLock` when locked methods call other locked methods of the same object.
- On an event loop, a critical section without `await` is already atomic. Add `asyncio.Lock` as soon as the section contains an `await`.

## Interview follow-ups

*The service now runs on four machines behind a load balancer. Does the lock still work?*

No. A `threading.Lock` is a data structure in one process's memory and is invisible to the other three machines. The invariant has to move to something all four share. The usual answers are a conditional update in the database, such as `UPDATE inventory SET units = units - 1 WHERE sku = ? AND units >= 1` with the affected row count as the success signal, or an atomic operation in a shared store such as Redis `DECR`, or a distributed lock with a lease and fencing token. The first is preferable when the data already lives in a transactional database, because it needs no extra infrastructure and cannot leave a lock stranded when a machine dies.

*How would you test for this bug in CI?*

Reproducing a race reliably is the hard part. Three techniques that work: run the operation from many threads and assert the invariant rather than any particular interleaving; insert a deliberate yield point at the suspected window, as `time.sleep(0)` does here, so the failure becomes deterministic; and use `sys.setswitchinterval` to shorten the interval between thread switches so that switches happen far more often than in normal operation.

*Reservations must expire after ten minutes if unpaid. Where does that fit?*

Expiry turns a single atomic operation into a state machine with a timer, and the timer thread mutates the same state the request threads mutate. It goes under the same per-SKU lock. This is also the point at which holding stock in memory stops being viable, because expiry has to survive a restart.

---

<a id="cs3"></a>
# Case Study 3: The Payment Gateway With a Ceiling

## The situation

A checkout service settles orders through an external payment provider. The contract permits at most **five concurrent connections**; a sixth simultaneous request is rejected with an error, and repeated violations get the merchant account throttled.

The nightly settlement job submits thirty charges through a thread pool sized at thirty, because thirty was faster than five in the developer's benchmark against the sandbox. In production the job produces a stream of rejections.

The cell below simulates the provider. It counts how many callers are inside `charge` at any instant and refuses anyone past the limit.

In [ ]:
import threading
import time
from concurrent.futures import ThreadPoolExecutor

from clab import timed


class TooManyConnections(Exception):
    pass


class PaymentGateway:
    """Simulates a provider that accepts at most `limit` concurrent requests."""

    def __init__(self, limit=5, latency=0.1):
        self.limit = limit
        self.latency = latency
        self._active = 0
        self.peak_concurrency = 0
        self.rejections = 0
        self._lock = threading.Lock()   # protects the counters, nothing else

    def charge(self, order_id):
        with self._lock:
            self._active += 1
            self.peak_concurrency = max(self.peak_concurrency, self._active)
            over_limit = self._active > self.limit
            if over_limit:
                self.rejections += 1
        try:
            if over_limit:
                raise TooManyConnections(f"order {order_id}: over {self.limit} connections")
            time.sleep(self.latency)
            return f"order {order_id} charged"
        finally:
            with self._lock:
                self._active -= 1


def settle(gateway, order_ids, workers):
    results, failures = [], []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(gateway.charge, oid) for oid in order_ids]
        for future in futures:
            try:
                results.append(future.result())
            except TooManyConnections as exc:
                failures.append(str(exc))
    return results, failures


orders = list(range(30))

gateway = PaymentGateway(limit=5)
with timed("v1  ThreadPoolExecutor(30), no limiting"):
    ok, failed = settle(gateway, orders, workers=30)

print(f"\ncharged            : {len(ok)}")
print(f"rejected           : {len(failed)}")
print(f"peak concurrency   : {gateway.peak_concurrency} (contract allows {gateway.limit})")

## Step 1: Why shrinking the pool is the wrong fix

The obvious response is `ThreadPoolExecutor(max_workers=5)`. It works, and it works for the wrong reason.

The limit belongs to the payment provider, not to this thread pool. As soon as a second code path calls the same provider, for example a refund worker or a retry job or an interactive checkout request, each path has its own pool and each pool independently believes it may use five connections. The aggregate breaks the contract while every individual component looks compliant. The pool size also silently couples an unrelated tuning decision, how much parallelism this batch job wants, to a correctness constraint imposed from outside.

What is needed is an object that represents the scarce resource itself, that every caller shares, and that grants permission to use it.

## Step 2: The semaphore

A **semaphore** is a counter with two operations. `acquire()` decrements the counter if it is greater than zero, and otherwise blocks until someone else increments it. `release()` increments the counter and wakes one waiter if any are waiting.

Initialised to five, it permits five holders at once and makes the sixth caller wait. A lock is the special case of a semaphore with a count of one, with the difference that a lock is conceptually owned by the thread that took it while a semaphore is not: any thread may release a permit that another thread acquired, which is occasionally useful and frequently the source of the bug shown in step 3.

Two properties make the semaphore the right model here. It is an **object**, so it can be created once next to the gateway client and shared by every caller. And it is **independent of the worker count**, so the batch job can keep thirty threads for other work while still holding at most five permits for the gateway.

In [ ]:
class RateLimitedGateway:
    """
    Wraps the provider client and owns the concurrency limit.
    Every caller in the process shares this object, so the limit holds across
    the whole application rather than per thread pool.
    """

    def __init__(self, gateway, max_concurrent=5):
        self._gateway = gateway
        self._permits = threading.Semaphore(max_concurrent)

    def charge(self, order_id):
        with self._permits:            # acquire on entry, release on exit
            return self._gateway.charge(order_id)


gateway = PaymentGateway(limit=5)
limited = RateLimitedGateway(gateway, max_concurrent=5)

with timed("v2  ThreadPoolExecutor(30) + Semaphore(5)"):
    ok, failed = settle(limited, orders, workers=30)

print(f"\ncharged            : {len(ok)}")
print(f"rejected           : {len(failed)}")
print(f"peak concurrency   : {gateway.peak_concurrency} (contract allows {gateway.limit})")

Thirty charges complete, nothing is rejected, and peak concurrency never exceeds five. Elapsed time is about six batches of five, which is the fastest schedule the contract permits.

The `with self._permits:` form matters for the same reason it mattered for locks. A permit released only on the success path is a permit lost forever, and after five exceptions the wrapper permits no traffic at all and the service hangs with no obvious cause.

## Step 3: `BoundedSemaphore` and the bug it catches

A plain `Semaphore` accepts more releases than acquires. Its counter simply grows, silently raising the limit above the intended value. The cause is usually a retry or cleanup path that releases a permit it never acquired, or a `finally` block that runs twice.

`threading.BoundedSemaphore` records its initial value and raises `ValueError` if the count ever rises above it, converting a silent capacity leak into an immediate, obvious failure. Prefer it as the default; there is no cost to the check and the failure it catches is very hard to diagnose otherwise.

In [ ]:
def leak_permits(semaphore, label):
    """A cleanup path that releases a permit it does not hold."""
    semaphore.acquire()
    semaphore.release()
    try:
        semaphore.release()          # the bug: one release too many
        print(f"{label}: the extra release was accepted, limit silently raised")
    except ValueError as exc:
        print(f"{label}: ValueError raised immediately ({exc})")


leak_permits(threading.Semaphore(5), "Semaphore       ")
leak_permits(threading.BoundedSemaphore(5), "BoundedSemaphore")

## Step 4: Concurrency limits are not rate limits

The two constraints look similar and are enforced by different mechanisms.

A **concurrency limit** bounds how many operations are in flight at the same instant. A semaphore enforces it exactly.

A **rate limit** bounds how many operations start within a period of time, such as ten requests per second. A semaphore cannot enforce this, because it has no notion of time. With five permits and a service that responds in one millisecond, a semaphore happily allows five thousand requests per second.

Most external APIs impose both, and the two need separate mechanisms.

The standard rate limiter is the **token bucket**. A bucket holds up to `capacity` tokens and refills at `rate` tokens per second. Each request removes one token, and a request that finds the bucket empty waits until enough time has passed for a token to appear. The bucket's capacity determines how large a burst is tolerated after an idle period, while the refill rate determines the sustained throughput. Setting capacity equal to rate allows a one-second burst; setting it to one produces perfectly even pacing.

In [ ]:
class TokenBucket:
    """Thread-safe token bucket. Allows `rate` operations per second on average."""

    def __init__(self, rate, capacity=None):
        self._rate = float(rate)
        self._capacity = float(capacity if capacity is not None else rate)
        self._tokens = self._capacity
        self._updated = time.monotonic()
        self._lock = threading.Lock()

    def acquire(self):
        while True:
            with self._lock:
                now = time.monotonic()
                # Refill in proportion to elapsed time, never above capacity.
                self._tokens = min(
                    self._capacity,
                    self._tokens + (now - self._updated) * self._rate,
                )
                self._updated = now
                if self._tokens >= 1.0:
                    self._tokens -= 1.0
                    return
                wait = (1.0 - self._tokens) / self._rate
            # Sleep outside the lock so other threads can refill and proceed.
            time.sleep(wait)

    def __enter__(self):
        self.acquire()
        return self

    def __exit__(self, *exc):
        return False


bucket = TokenBucket(rate=10, capacity=10)   # 10 per second, burst of 10
timestamps = []


def paced_call(i):
    with bucket:
        timestamps.append(time.monotonic())


start = time.monotonic()
with ThreadPoolExecutor(max_workers=20) as pool:
    list(pool.map(paced_call, range(30)))
elapsed = time.monotonic() - start

timestamps.sort()
print(f"30 calls in {elapsed:.2f}s")
print(f"burst      : first 10 calls in {timestamps[9]-timestamps[0]:.2f}s (the bucket started full)")
print(f"sustained  : {20/(timestamps[29]-timestamps[9]):.1f} per second afterwards (configured 10/s)")

The first ten calls drain the bucket almost instantly, which is the burst allowance, and the remaining twenty are paced at ten per second. Total elapsed time is therefore about two seconds rather than three, and the sustained rate converges on the configured value.

Note that the `time.sleep(wait)` sits deliberately outside the `with self._lock:` block. Sleeping while holding a lock blocks every other thread from even checking the bucket, which would serialise the waiting and destroy the throughput the limiter is supposed to shape. **Never hold a lock across a blocking call** unless the blocking call is itself the thing being protected.

## Step 5: The same pattern in asyncio

`asyncio.Semaphore` has the same semantics with a coroutine-friendly implementation: a waiting task suspends and the loop runs others, instead of blocking the thread.

This produces the standard idiom for bounded fan-out in asyncio. Create every task at once, since tasks are cheap, and let a semaphore inside the worker decide how many actually proceed. Ten thousand pending tasks sharing five permits is entirely reasonable, and the equivalent with threads would be ten thousand operating system threads.

In [ ]:
import asyncio

from clab import run_async


class AsyncGateway:
    """Async version of the provider, with the same concurrency ceiling."""

    def __init__(self, limit=5, latency=0.1):
        self.limit = limit
        self.latency = latency
        self._active = 0
        self.peak_concurrency = 0
        self.rejections = 0

    async def charge(self, order_id):
        self._active += 1             # no lock needed: no await between these lines
        self.peak_concurrency = max(self.peak_concurrency, self._active)
        try:
            if self._active > self.limit:
                self.rejections += 1
                raise TooManyConnections(f"order {order_id}")
            await asyncio.sleep(self.latency)
            return f"order {order_id} charged"
        finally:
            self._active -= 1


async def settle_async(gateway, order_ids, max_concurrent=None):
    if max_concurrent is None:
        async def worker(oid):
            return await gateway.charge(oid)
    else:
        permits = asyncio.Semaphore(max_concurrent)

        async def worker(oid):
            async with permits:
                return await gateway.charge(oid)

    return await asyncio.gather(
        *(worker(oid) for oid in order_ids), return_exceptions=True
    )


gw = AsyncGateway(limit=5)
results = run_async(settle_async(gw, orders))
failures = [r for r in results if isinstance(r, Exception)]
print(f"no semaphore     : {len(failures)} rejected, peak concurrency {gw.peak_concurrency}")

gw = AsyncGateway(limit=5)
results = run_async(settle_async(gw, orders, max_concurrent=5))
failures = [r for r in results if isinstance(r, Exception)]
print(f"asyncio.Semaphore: {len(failures)} rejected, peak concurrency {gw.peak_concurrency}")

Two details in that cell are worth naming.

`return_exceptions=True` makes `gather` collect exceptions as ordinary results rather than propagating the first one and leaving the remaining tasks in an ambiguous state. For a batch where partial success is meaningful, this is usually what you want. Case Study 6 examines the alternatives.

The counter increments in `AsyncGateway.charge` need no lock because no `await` separates the read from the write, which is the guarantee established in Case Study 2. Adding an `await` between those two lines would reintroduce the race.

## Takeaways

- A semaphore is a counting permit system. Use it when a limited resource is shared by several call sites, so the limit lives with the resource rather than with any one pool.
- `with semaphore:` guarantees the permit is returned even on failure. Permits leaked on error paths lead to a silent, permanent stall.
- Prefer `BoundedSemaphore`, which turns an over-release into an immediate `ValueError` instead of a silently raised limit.
- Concurrency limits and rate limits are different constraints. Semaphores enforce the first; token buckets or sliding windows enforce the second.
- Never block while holding a lock, and be careful to release the lock before sleeping inside a limiter.
- In asyncio, create all the tasks and let a semaphore bound how many run. Tasks are cheap enough that this is the natural structure.

## Interview follow-ups

*The limit is five per merchant, and you serve a thousand merchants. What changes?*

One semaphore per merchant, held in a dictionary keyed by merchant id, created lazily under a lock that guards the dictionary itself. The next question is eviction, since a thousand rarely used semaphores are harmless but a million would not be, and a semaphore with waiters must never be evicted. A bounded LRU with a check that the semaphore is idle before removal is the usual answer.

*Requests spend most of their time waiting on the provider. Why not use asyncio for everything here?*

That is often right. The obstacle is usually the client library: if the provider's SDK is synchronous, calling it from a coroutine blocks the loop and destroys the model's advantage. The choices are an async-native client, or wrapping the synchronous client with `asyncio.to_thread`, which reintroduces threads and their cost for that call.

*How do you handle the provider returning 429 despite your limiter?*

Treat your limiter as an optimisation and the provider's response as the authority. Retry with exponential backoff and jitter, honour a `Retry-After` header when present, and give up after a bounded number of attempts. Jitter matters: without it, every client retries at the same moment and reproduces the burst that caused the rejection.

---

<a id="cs4"></a>
# Case Study 4: The Log Ingestion Pipeline

## The situation

A platform team runs a service that receives application log lines over the network, parses each one, and writes batches to a search index. The rates on the two sides are very different: log lines arrive in bursts of thousands, while the index accepts a batch every few tens of milliseconds.

The first implementation uses a shared list as a hand-off point between a producer thread and several consumer threads. It works in development, where one consumer keeps up easily. In production it throws `IndexError` from a line that appears to have just checked that the list is not empty, and during traffic spikes the process grows to several gigabytes and is killed by the container runtime.

Both symptoms come from the same design gap. Read the code and identify them before running it.

In [ ]:
import threading
import time
from collections import deque


class NaivePipeline:
    """A plain list used as a hand-off point. Two defects, both structural."""

    def __init__(self):
        self.buffer = []
        self.processed = 0
        self.errors = []
        self._running = True

    def produce(self, lines):
        for line in lines:
            self.buffer.append(line)     # never waits, never asks how full it is

    def consume(self):
        while self._running:
            if self.buffer:                       # 1. check that it is not empty
                try:
                    time.sleep(0)                 #    a switch may happen here
                    line = self.buffer.pop(0)     # 2. act on a stale observation
                except IndexError as exc:
                    self.errors.append(exc)
                    continue
                self.processed += 1
            else:
                time.sleep(0.001)                 # busy waiting burns CPU


pipeline = NaivePipeline()
consumers = [threading.Thread(target=pipeline.consume, daemon=True) for _ in range(4)]
for c in consumers:
    c.start()

pipeline.produce([f"line-{i}" for i in range(2000)])
time.sleep(0.5)
pipeline._running = False
for c in consumers:
    c.join(timeout=1)

print(f"lines produced  : 2000")
print(f"lines processed : {pipeline.processed}")
print(f"IndexErrors     : {len(pipeline.errors)}  (check-then-act on an empty list)")

## Step 1: Two distinct defects

**The race.** `if self.buffer:` followed by `self.buffer.pop(0)` is the check-then-act pattern from Case Study 2, wearing different clothes. Four consumers can all observe a single remaining item, and three of them will find the list empty by the time they pop. Every consumer also busy-waits when the list is empty, burning CPU on a loop that does nothing.

**The absence of backpressure.** `produce` appends without limit and never asks whether the consumers are keeping up. When arrivals outpace processing, the difference accumulates in memory until the process is killed. **Backpressure** is the mechanism by which a slow consumer forces a fast producer to slow down, and a pipeline without it converts a temporary rate mismatch into an unbounded memory leak.

There is a third, smaller defect worth spotting: `self.processed += 1` is an unsynchronised read, add and write executed by four threads, so the count itself can lose updates. It is the Case Study 2 bug hiding inside the Case Study 4 bug.

The first two defects have the same fix, because both are properties that a proper queue already has.

## Step 2: `queue.Queue`

`queue.Queue` is a thread-safe first-in, first-out queue. It solves both problems and removes the busy-wait as a side effect.

- `put(item)` adds an item, blocking if the queue is full.
- `get()` removes and returns an item, blocking until one is available. The waiting thread sleeps rather than spinning, so it consumes no CPU.
- `maxsize` caps the number of queued items. Once full, `put` blocks, and the producer is throttled to the consumers' pace. This is backpressure, implemented in one constructor argument.
- `task_done()` marks a retrieved item as finished, and `join()` blocks until every item that was put has been marked done. Together they express "wait until the backlog is drained" without polling.

The check-then-act race cannot occur because `get()` is a single atomic operation rather than a check followed by an act. Internally the queue uses a lock and two condition variables, which is exactly the machinery built by hand in Case Study 5, done correctly once.

**Shutting workers down.** A worker blocked in `get()` cannot notice a flag, because it is not running. The standard technique is a **sentinel**, a distinguishable value that means "no more work". One sentinel is needed per worker, since each is consumed by exactly one of them.

In [ ]:
import queue
from clab import timed


def parse(line):
    """Stand-in for parsing and enriching a log line."""
    time.sleep(0.001)
    return line.upper()


SENTINEL = object()   # a unique marker; no real log line can equal it


def run_pipeline(lines, workers=4, maxsize=100):
    work_queue = queue.Queue(maxsize=maxsize)
    results = []
    results_lock = threading.Lock()

    def consumer():
        while True:
            item = work_queue.get()          # blocks; no polling, no CPU burn
            try:
                if item is SENTINEL:
                    return
                parsed = parse(item)
                with results_lock:
                    results.append(parsed)
            finally:
                # In a finally block so the count stays correct even if parse
                # raises. Step 4 shows what happens when this is forgotten.
                work_queue.task_done()

    threads = [threading.Thread(target=consumer) for _ in range(workers)]
    for t in threads:
        t.start()

    for line in lines:
        work_queue.put(line)                 # blocks when the queue is full

    work_queue.join()                        # wait until the backlog is drained

    for _ in threads:                        # one sentinel per worker
        work_queue.put(SENTINEL)
    for t in threads:
        t.join()

    return results


lines = [f"line-{i}" for i in range(400)]

with timed("queue.Queue with 4 workers"):
    results = run_pipeline(lines, workers=4)

print(f"\nlines produced  : {len(lines)}")
print(f"lines processed : {len(results)}")
print(f"errors          : 0")

Every line is processed, nothing is lost, no exception is raised, and the workers exit cleanly.

## Step 3: Backpressure, measured

The `maxsize` argument is easy to overlook and is the difference between a pipeline that degrades gracefully and one that runs out of memory. The next cell instruments the queue depth while a fast producer feeds slow consumers, first without a bound and then with one.

In [ ]:
def measure_depth(maxsize, items=600, workers=2, consume_cost=0.002):
    work_queue = queue.Queue(maxsize=maxsize)   # 0 means unbounded
    peak = 0
    stop = object()

    def consumer():
        while True:
            item = work_queue.get()
            try:
                if item is stop:
                    return
                time.sleep(consume_cost)
            finally:
                work_queue.task_done()

    threads = [threading.Thread(target=consumer) for _ in range(workers)]
    for t in threads:
        t.start()

    start = time.perf_counter()
    for i in range(items):
        work_queue.put(i)
        peak = max(peak, work_queue.qsize())
    produce_time = time.perf_counter() - start

    work_queue.join()
    for _ in threads:
        work_queue.put(stop)
    for t in threads:
        t.join()
    return peak, produce_time, time.perf_counter() - start


peak, produce_time, total = measure_depth(maxsize=0)
print(f"unbounded queue : peak depth {peak:4d} items, producer finished in {produce_time:.2f}s, total {total:.2f}s")

peak, produce_time, total = measure_depth(maxsize=20)
print(f"maxsize=20      : peak depth {peak:4d} items, producer finished in {produce_time:.2f}s, total {total:.2f}s")

The unbounded queue lets the producer finish almost immediately, leaving the entire backlog resident in memory. The bounded queue makes the producer wait once the queue is full, so its completion time tracks the consumers' pace and memory stays flat.

Total elapsed time is essentially the same in both cases, because the consumers are the bottleneck either way. What changes is the peak memory and, in a real system, what happens upstream. With backpressure the slowdown propagates: the socket read stalls, the TCP receive window shrinks, and the sender is throttled. That is a system behaving correctly under overload. Without backpressure, the same overload is absorbed silently until the process dies, taking the entire backlog with it.

Choosing `maxsize` is a capacity decision. Large enough to absorb normal bursts without stalling the producer, small enough that the worst case fits in memory, and small enough that an item does not sit in the queue longer than its useful lifetime.

## Step 4: The worker that dies quietly

A worker that raises an exception simply ends. The thread disappears, `task_done()` is never called for the item it was holding, and the pipeline is quietly down one worker. If the main thread is waiting in `queue.join()`, it waits forever, because the internal count of unfinished tasks never reaches zero.

This produces one of the most confusing production symptoms in this whole area: a service that stops making progress, with no error, no crash, and no obvious culprit. The next cell reproduces it with a watchdog so the notebook does not hang.

In [ ]:
def fragile_pipeline(items=20, poison_at=5, timeout=2.0):
    work_queue = queue.Queue()

    def consumer():
        while True:
            item = work_queue.get()
            if item is None:
                work_queue.task_done()
                return
            if item == poison_at:
                raise ValueError("bad record")   # thread dies here, holding an item
            work_queue.task_done()

    worker = threading.Thread(target=consumer, daemon=True)
    worker.start()
    for i in range(items):
        work_queue.put(i)

    finished = threading.Event()
    threading.Thread(target=lambda: (work_queue.join(), finished.set()), daemon=True).start()

    if finished.wait(timeout):
        print("pipeline drained")
    else:
        print(f"HUNG: queue.join() has waited {timeout}s and will wait forever.")
        print(f"      unfinished tasks remaining : {work_queue.unfinished_tasks}")
        print(f"      worker alive               : {worker.is_alive()}")


fragile_pipeline()

Three rules follow, and all three belong in any worker loop you write.

**Call `task_done()` in a `finally` block.** The bookkeeping must be correct regardless of what the handler does.

**Catch exceptions inside the worker.** A worker loop should treat a failing item as data, not as a reason to terminate. Log it, count it, route it to a dead-letter queue, and continue.

**Distinguish a poison message from a worker fault.** One malformed record should not remove a worker from the pool. If the failure is genuinely fatal, such as a lost database connection, shut the pipeline down deliberately rather than by attrition.

The corrected worker below also demonstrates a shutdown mechanism that does not rely on sentinels: an `Event` combined with `get(timeout=...)`. This suits a long-running service, where shutdown is triggered by a signal rather than by the end of a finite input.

In [ ]:
class LogPipeline:
    """A worker pool that survives bad records and shuts down on request."""

    def __init__(self, workers=4, maxsize=100):
        self._queue = queue.Queue(maxsize=maxsize)
        self._shutdown = threading.Event()
        self._lock = threading.Lock()
        self.processed = 0
        self.failed = []
        self._threads = [
            threading.Thread(target=self._worker, name=f"worker-{i}")
            for i in range(workers)
        ]
        for t in self._threads:
            t.start()

    def _worker(self):
        while not self._shutdown.is_set():
            try:
                # The timeout gives the loop a chance to re-check the flag.
                item = self._queue.get(timeout=0.1)
            except queue.Empty:
                continue
            try:
                self._handle(item)
                with self._lock:
                    self.processed += 1
            except Exception as exc:              # one bad record, not a dead worker
                with self._lock:
                    self.failed.append((item, repr(exc)))
            finally:
                self._queue.task_done()

    def _handle(self, item):
        if "corrupt" in item:
            raise ValueError(f"cannot parse {item!r}")
        time.sleep(0.001)

    def submit(self, item):
        self._queue.put(item)                     # blocks when full: backpressure

    def drain(self):
        self._queue.join()

    def stop(self):
        self.drain()
        self._shutdown.set()
        for t in self._threads:
            t.join(timeout=2)


pipeline = LogPipeline(workers=4)
for i in range(300):
    pipeline.submit(f"corrupt-{i}" if i % 50 == 0 else f"line-{i}")
pipeline.stop()

print(f"processed        : {pipeline.processed}")
print(f"failed records   : {len(pipeline.failed)}")
print(f"example failure  : {pipeline.failed[0]}")
print(f"workers alive    : {sum(t.is_alive() for t in pipeline._threads)}")

## Step 5: The asyncio equivalent

`asyncio.Queue` offers the same interface with coroutine semantics: `await q.put(item)` and `await q.get()` suspend the calling task instead of blocking the thread. The structure of the solution is unchanged, which is a useful signal that producer and consumer is a genuine architectural pattern rather than a threading trick.

One difference matters for shutdown. An asyncio worker waiting in `await q.get()` is a task, and tasks can be cancelled. `task.cancel()` raises `CancelledError` inside the waiting task, which is a cleaner shutdown than sentinels for many designs. The version below keeps sentinels so the comparison with the threaded code is direct, and Case Study 6 covers cancellation properly.

In [ ]:
import asyncio

from clab import run_async


async def async_pipeline(lines, workers=4, maxsize=50):
    work_queue = asyncio.Queue(maxsize=maxsize)
    results = []

    async def consumer():
        while True:
            item = await work_queue.get()
            try:
                if item is None:
                    return
                await asyncio.sleep(0.001)        # the async equivalent of parse()
                results.append(item.upper())
            finally:
                work_queue.task_done()

    consumers = [asyncio.create_task(consumer()) for _ in range(workers)]

    for line in lines:
        await work_queue.put(line)                # suspends when full: backpressure

    await work_queue.join()
    for _ in consumers:
        await work_queue.put(None)
    await asyncio.gather(*consumers)
    return results


with timed("asyncio.Queue with 4 consumer tasks"):
    results = run_async(async_pipeline([f"line-{i}" for i in range(400)]))

print(f"\nprocessed {len(results)} lines on a single thread")

## Queue or executor?

Both `queue.Queue` with worker threads and `ThreadPoolExecutor` distribute work across workers, and an executor is in fact a queue with a pool attached. Choosing between them is a question of what the code needs to control.

| | `ThreadPoolExecutor` | `queue.Queue` with worker threads |
|---|---|---|
| Best for | a finite batch of independent tasks | a long-running service consuming a stream |
| Results | returned as `Future` objects | you design the result path |
| Backpressure | none by default; `submit` always accepts | `maxsize` gives it directly |
| Lifetime | bounded by the `with` block | runs until told to stop |
| Multi-stage pipelines | awkward | natural, one queue between stages |
| Code volume | minimal | more, in exchange for control |

Use the executor when the work is a batch with a beginning and an end. Use an explicit queue when the work is a continuous stream, when you need backpressure, or when the pipeline has several stages with different worker counts, such as four parsers feeding one writer.

## Takeaways

- A list plus a lock is a queue that you have to get right yourself. `queue.Queue` is already correct, already thread-safe, and does not busy-wait.
- `maxsize` is backpressure. Without it, a rate mismatch becomes an unbounded memory leak.
- `task_done()` belongs in a `finally` block, and worker loops must catch exceptions, or the pool silently loses workers and `join()` never returns.
- Sentinels shut down a queue driven by finite input. `Event` plus `get(timeout=...)` suits a long-running service.
- `asyncio.Queue` mirrors the same design for coroutines, with cancellation as an additional shutdown route.

## Interview follow-ups

*The index is unavailable for thirty seconds. What should the pipeline do?*

Backpressure will propagate the stall upstream, which is correct behaviour but not sufficient by itself. Add retries with exponential backoff around the write, a bounded number of attempts, and a dead-letter queue for records that exhaust them. Add a circuit breaker so that when the index is clearly down the pipeline fails fast instead of holding thousands of threads in retry loops. Decide explicitly whether the input source can be paused, as with a Kafka consumer that simply stops committing offsets, or whether data will be dropped, and make the choice visible in a metric.

*Ordering must be preserved per tenant. How does that change the design?*

A single queue with four workers gives no ordering guarantee across workers. Partition instead: hash the tenant id to one of N queues, each drained by exactly one worker. Ordering holds within a partition, which is what the requirement asks for, and this is the same design Kafka uses. The cost is that one heavy tenant can now saturate its own partition while others idle.

*Why not use `multiprocessing.Queue` and run consumers in processes?*

If parsing is CPU-bound in pure Python, that is exactly the right move, and it is the pattern in Case Study 1. The cost is that every item is pickled and copied across a pipe, so items should be large enough to justify the transfer, and the queue no longer offers the convenience of shared result lists.

---

<a id="cs5"></a>
# Case Study 5: The Cache That Stampeded

## The situation

A product service caches category listings in memory because computing one costs about 300 milliseconds of database work. The cache works well until a popular entry expires during peak traffic. At that moment every in-flight request finds the entry missing, and every one of them starts the same 300 millisecond computation. The database receives forty identical queries, its latency rises, the computations take longer, more requests pile up, and the service falls over.

This failure has a name: **cache stampede**, also called the thundering herd. The cell below reproduces it and counts how many times the loader actually ran.

In [ ]:
import threading
import time

from clab import timed


class LoaderStats:
    """Counts how many times the expensive loader was invoked."""

    def __init__(self, cost=0.3):
        self.calls = 0
        self.cost = cost
        self._lock = threading.Lock()

    def load(self, key):
        with self._lock:
            self.calls += 1
        time.sleep(self.cost)               # the expensive database query
        return f"value-for-{key}"


class NaiveCache:
    def __init__(self, loader):
        self._data = {}
        self._loader = loader

    def get(self, key):
        if key in self._data:              # 1. check
            return self._data[key]
        value = self._loader(key)          # 2. everybody who missed loads it
        self._data[key] = value
        return value


def stampede(cache, key="category:electronics", callers=40):
    threads = [threading.Thread(target=cache.get, args=(key,)) for _ in range(callers)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()


stats = LoaderStats()
cache = NaiveCache(stats.load)

with timed("40 concurrent requests, naive cache"):
    stampede(cache)

print(f"\nloader invocations : {stats.calls}  (the ideal is 1)")

## Step 1: Single flight with a per-key lock

The property we want is called **single flight**: for a given key, at most one load is in progress, and every other caller that arrives while it runs receives the result of that load rather than starting its own.

A lock provides it, with two details that are easy to get wrong.

**The lock must be per key.** A single global lock would give single flight and would also mean that a slow load of one key blocks every request for every other key for 300 milliseconds. Locking per key confines the effect to the key being loaded, which is the same granularity argument as the per-SKU lock in Case Study 2.

**The check must be repeated inside the lock.** The first check outside the lock is the fast path, so that a cache hit costs no lock acquisition at all. Threads that miss then take the lock, and by the time a waiting thread acquires it the winner has usually already stored the value. Checking again turns those waiters into hits. Without the second check, every waiter runs the loader in turn, and the stampede is serialised rather than eliminated. This structure is called **double-checked locking**, and it is the same pattern used for a thread-safe lazy singleton.

The dictionary of per-key locks is itself shared mutable state, so it needs its own small lock to guard creation of new entries.

In [ ]:
class SingleFlightCache:
    def __init__(self, loader):
        self._data = {}
        self._loader = loader
        self._key_locks = {}
        self._registry_lock = threading.Lock()   # guards _key_locks only

    def _lock_for(self, key):
        with self._registry_lock:
            lock = self._key_locks.get(key)
            if lock is None:
                lock = self._key_locks[key] = threading.Lock()
            return lock

    def get(self, key):
        # Fast path: a hit costs no lock at all.
        if key in self._data:
            return self._data[key]

        with self._lock_for(key):
            # Second check: the thread that held the lock before us has
            # probably already stored the value.
            if key in self._data:
                return self._data[key]
            value = self._loader(key)
            self._data[key] = value
            return value


stats = LoaderStats()
cache = SingleFlightCache(stats.load)

with timed("40 concurrent requests, single flight"):
    stampede(cache)

print(f"\nloader invocations : {stats.calls}")

# Different keys must not block one another.
stats = LoaderStats()
cache = SingleFlightCache(stats.load)
keys = [f"category:{i}" for i in range(4)]
threads = [threading.Thread(target=cache.get, args=(k,)) for k in keys for _ in range(10)]

with timed("40 requests across 4 distinct keys"):
    for t in threads:
        t.start()
    for t in threads:
        t.join()

print(f"\nloader invocations : {stats.calls}  (one per distinct key)")

One load instead of forty, and the total time is one load rather than forty. Four distinct keys produce four loads that run concurrently rather than in sequence, which confirms that the granularity is right.

## Step 2: `Event`, for announcing that something has happened

A lock answers the question "may I proceed alone?" A different question comes up constantly in services: "has this thing happened yet?" Worker threads must not serve traffic until configuration has loaded. A retry loop must stop when shutdown is requested. A test must wait for a background thread to reach a known state.

`threading.Event` is the primitive for that question. It holds a single boolean flag with three operations: `set()` raises the flag, `clear()` lowers it, and `wait(timeout=None)` blocks until the flag is raised, returning `True` if it was raised and `False` if the timeout expired. `is_set()` reads it without waiting.

Two properties distinguish it from a lock. It is a **broadcast**: `set()` releases every waiter at once, not one of them. And it is **sticky**: once set, later calls to `wait()` return immediately, so a thread that arrives after the event does not wait forever for an announcement it missed. That second property is what makes `Event` safe against the ordering problem that catches people using condition variables by hand.

Always pass a timeout to `wait()` in production code. `wait()` with no timeout is an unbounded hang if the setter dies before setting.

In [ ]:
class ConfigService:
    """Workers must not serve traffic until configuration has loaded."""

    def __init__(self):
        self.config = None
        self.ready = threading.Event()
        self.served = []
        self._lock = threading.Lock()

    def load_config(self, delay=0.3):
        time.sleep(delay)                     # fetch from a config server
        self.config = {"region": "ap-south-1", "timeout_ms": 250}
        self.ready.set()                      # release every waiting worker at once

    def serve(self, worker_id):
        if not self.ready.wait(timeout=2.0):  # always bound the wait
            with self._lock:
                self.served.append((worker_id, "timed out waiting for config"))
            return
        with self._lock:
            self.served.append((worker_id, self.config["region"]))


service = ConfigService()
workers = [threading.Thread(target=service.serve, args=(i,)) for i in range(6)]
for w in workers:
    w.start()

time.sleep(0.05)
print(f"config ready after 50ms? {service.ready.is_set()}")
print(f"workers served so far  : {len(service.served)}")

threading.Thread(target=service.load_config).start()
for w in workers:
    w.join()

print(f"\nconfig ready now?      {service.ready.is_set()}")
print(f"workers served         : {len(service.served)}")
print(f"sample                 : {service.served[:3]}")

late = ConfigService()
late.config = {"region": "eu-west-1"}
late.ready.set()
late.serve(99)
print(f"\na worker arriving after the event: {late.served[-1]} (no waiting, the event is sticky)")

## Step 3: `Condition`, for waiting on a predicate

`Event` handles a flag that is raised once. Many situations need a thread to wait until an arbitrary condition over shared state becomes true, where that condition can flip back and forth: wait until the buffer has room, wait until at least one connection is free, wait until the queue is not empty.

`threading.Condition` combines a lock with a waiting area. A thread acquires the lock, inspects the shared state, and if the state is not yet suitable it calls `wait()`. `wait()` atomically releases the lock and suspends the thread, which is the essential trick: releasing the lock lets another thread change the state, and doing it atomically with the suspension closes the window in which a notification could be missed. When another thread changes the state and calls `notify()` or `notify_all()`, a waiter wakes, reacquires the lock, and re-inspects.

The predicate must be tested in a `while` loop rather than an `if`. Two reasons, both real:

- `notify_all()` wakes every waiter, but the lock admits them one at a time. The first waiter may consume the only available item, so by the time the second one holds the lock the condition is false again.
- Waiters may wake without a corresponding notification. Rechecking makes the code correct regardless.

The rule is short and worth memorising: **wait inside a `while` that tests the predicate**. Python provides `wait_for(predicate)` which writes the loop for you.

The bounded buffer below is the classic example, and it is essentially what `queue.Queue` implements internally. Building it once by hand makes the queue's guarantees concrete.

In [ ]:
from collections import deque


class BoundedBuffer:
    """A bounded FIFO built by hand from a Condition. This is queue.Queue in miniature."""

    def __init__(self, capacity):
        self._items = deque()
        self._capacity = capacity
        self._cond = threading.Condition()      # creates its own internal lock
        self.peak = 0

    def put(self, item):
        with self._cond:
            while len(self._items) >= self._capacity:   # while, never if
                self._cond.wait()
            self._items.append(item)
            self.peak = max(self.peak, len(self._items))
            self._cond.notify_all()                     # a consumer may now proceed

    def get(self):
        with self._cond:
            while not self._items:
                self._cond.wait()
            item = self._items.popleft()
            self._cond.notify_all()                     # a producer may now proceed
            return item


buffer = BoundedBuffer(capacity=5)
consumed = []
DONE = object()


def producer():
    for i in range(60):
        buffer.put(i)
    for _ in range(3):
        buffer.put(DONE)


def consumer():
    while True:
        item = buffer.get()
        if item is DONE:
            return
        time.sleep(0.002)
        consumed.append(item)


threads = [threading.Thread(target=producer)] + [
    threading.Thread(target=consumer) for _ in range(3)
]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"items consumed  : {len(consumed)}")
print(f"peak buffer size: {buffer.peak} (capacity 5, so the producer was throttled)")

`notify()` wakes one waiter and `notify_all()` wakes all of them. `notify()` is cheaper, and it is only safe when every waiter is waiting for the same condition and any one of them can make progress. When producers and consumers wait on the same condition object, as they do above, `notify()` can wake a thread that cannot proceed and leave the one that could still asleep, which stalls the system. Use `notify_all()` unless you have reasoned carefully about who is waiting for what.

The broader point: this is exactly why `queue.Queue` exists. Nearly every producer and consumer problem should use the queue, and understanding `Condition` is what lets you read the queue's source, debug it, and recognise the handful of cases where a custom predicate genuinely requires building your own.

## Step 4: Two smaller primitives worth knowing

**`Barrier`** blocks a fixed number of threads until all of them have arrived, then releases them together. It is used for phase synchronisation, where every worker must finish phase one before any starts phase two, and it is the correct way to start a benchmark so that measurement is not distorted by thread startup stagger.

**`threading.local`** provides storage whose contents are private to each thread. An attribute set on a `threading.local` object in one thread is invisible to every other thread. This is how per-thread resources that are expensive to create and unsafe to share, such as database connections, HTTP sessions, and random number generators, are managed without passing them through every function signature. Web frameworks use it for request context.

In [ ]:
import random

# Barrier: make every worker begin the measured phase at the same instant.
def measure_start_spread(use_barrier, seed=1):
    n = 8
    barrier = threading.Barrier(n)
    starts = []
    lock = threading.Lock()
    rng = random.Random(seed)
    setup_costs = [rng.uniform(0, 0.05) for _ in range(n)]

    def worker(i):
        time.sleep(setup_costs[i])  # workers need different amounts of setup
        if use_barrier:
            barrier.wait()          # nobody proceeds until all 8 have arrived
        with lock:
            starts.append(time.perf_counter())

    threads = [threading.Thread(target=worker, args=(i,)) for i in range(n)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return (max(starts) - min(starts)) * 1000


print(f"start spread without barrier : {measure_start_spread(False):6.2f} ms")
print(f"start spread with barrier    : {measure_start_spread(True):6.2f} ms")

# threading.local: one connection object per thread, created on first use.
connections = threading.local()
created = []


def query(sql):
    if not hasattr(connections, "conn"):
        # Expensive to create and unsafe to share, so build one per thread.
        connections.conn = f"connection-for-{threading.current_thread().name}"
        created.append(connections.conn)
    time.sleep(0.02)                     # the query itself
    return f"{connections.conn} ran {sql}"


from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=3, thread_name_prefix="db") as pool:
    list(pool.map(query, [f"SELECT {i}" for i in range(12)]))

print(f"\n12 queries across 3 threads created {len(created)} connections:")
for c in created:
    print(f"  {c}")

## Takeaways

- Cache stampede is a check-then-act race with an expensive act. Single flight, implemented as a per-key lock with a double check, reduces N loads to one.
- Check outside the lock for speed, check again inside the lock for correctness. Both checks are needed.
- `Event` answers "has this happened yet?" It broadcasts to all waiters and is sticky, so late arrivals do not miss the announcement. Always pass a timeout to `wait()`.
- `Condition` answers "has the state become suitable yet?" Wait inside a `while` loop that tests the predicate, and prefer `notify_all()` unless every waiter is interchangeable.
- `queue.Queue` is a correct `Condition`-based bounded buffer. Use it rather than writing your own.
- `Barrier` synchronises phases and produces fair benchmark starts. `threading.local` gives each thread its own copy of a resource that is expensive to create and unsafe to share.

## Interview follow-ups

*A per-key lock still means one caller waits 300 milliseconds while others wait behind it. Can that be improved?*

Yes, with **stale-while-revalidate**. Keep the expired value and serve it immediately to every caller while one background task refreshes the entry. Nobody waits, at the cost of serving data that is briefly out of date. Add a soft expiry, at which refresh begins, ahead of the hard expiry, at which the value is no longer served, and add jitter to the expiry times so that a thousand keys written together do not all expire in the same second and recreate the stampede at a larger scale.

*The cache is now Redis, shared by twenty application servers. Does single flight still work?*

The per-key `threading.Lock` reduces the stampede from twenty times forty callers to twenty, one per server, which is a large improvement and not a solution. Making it one requires a lock all twenty servers can see, typically `SET key value NX PX ttl` in Redis, where the process that wins the write performs the load and the others poll briefly or serve stale data. The lock must carry a time to live so that a crashed holder does not block the key forever.

*How would you test the stampede fix?*

Assert on the number of loader invocations rather than on timing. Instrument the loader with a counter, fire N concurrent requests for a cold key, and assert the counter equals one. The test is deterministic and does not depend on machine speed, which is what makes it safe to run in CI.

---

<a id="cs6"></a>
# Case Study 6: The Dashboard That Missed Its Deadline

## The situation

An account dashboard assembles one screen from four internal services: profile, billing, usage, and recommendations. The product requirement is that the endpoint responds within one second, and that a failure in recommendations degrades the screen rather than breaking it.

The current implementation calls the four services in sequence. Their latencies add up, the recommendations service occasionally fails, and when it does the whole endpoint returns a 500.

This case study is about orchestration: running several operations, collecting their results and their failures, enforcing a deadline, and doing all of it without letting a slow or broken dependency take the endpoint down.

In [ ]:
import random
import time

from clab import timed


class ServiceError(Exception):
    pass


def call_service(name, latency, fail=False):
    """Simulates one internal HTTP call."""
    time.sleep(latency)
    if fail:
        raise ServiceError(f"{name} is unavailable")
    return {name: f"data from {name}"}


SERVICES = [
    ("profile", 0.20, False),
    ("billing", 0.35, False),
    ("usage", 0.30, False),
    ("recommendations", 0.25, True),   # this one is broken today
]


def dashboard_sequential():
    screen = {}
    for name, latency, fail in SERVICES:
        screen.update(call_service(name, latency, fail))
    return screen


with timed("v1  sequential fan-out"):
    try:
        screen = dashboard_sequential()
    except ServiceError as exc:
        print(f"\nendpoint failed: {exc}")
        print("Three healthy services were called and their results discarded.")

## Step 1: Fan out with futures

The four calls are independent, so they should overlap. `ThreadPoolExecutor.submit` schedules one callable and immediately returns a **`Future`**: an object representing a result that does not exist yet.

A `Future` supports the operations you need to orchestrate work:

- `result(timeout=None)` blocks until the value is available and returns it. If the callable raised, `result()` re-raises that exception in the calling thread, with the original traceback attached.
- `exception(timeout=None)` returns the exception instead of raising it.
- `done()` reports whether the task has finished, without waiting.
- `cancel()` attempts to cancel, and succeeds only if the task has not started yet.

`concurrent.futures.as_completed` takes an iterable of futures and yields each one as it finishes, in completion order rather than submission order. That property is what makes it possible to enforce a deadline over the group and to start using early results before the slow ones arrive.

The idiom below maps each future back to its service name with a dictionary, because `as_completed` gives you a future and no other context.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def dashboard_parallel():
    screen = {}
    degraded = []

    with ThreadPoolExecutor(max_workers=4) as pool:
        # Remember which future belongs to which service.
        futures = {
            pool.submit(call_service, name, latency, fail): name
            for name, latency, fail in SERVICES
        }
        for future in as_completed(futures):
            name = futures[future]
            try:
                screen.update(future.result())     # re-raises whatever the task raised
            except ServiceError as exc:
                degraded.append(f"{name}: {exc}")

    return screen, degraded


with timed("v2  parallel fan-out, failures isolated"):
    screen, degraded = dashboard_parallel()

print(f"\nsections rendered : {sorted(screen)}")
print(f"degraded sections : {degraded}")

Elapsed time is now the slowest call rather than the sum of all four, and the failure is contained to one section of the screen instead of the whole response.

## Step 2: The exception that disappears

A `Future` stores the exception raised by its task and does nothing with it until someone asks. Code that submits work and never calls `result()` or `exception()` loses the failure entirely: no traceback, no log line, no crash. A background task that has been silently dying for weeks is a common and expensive discovery.

`pool.map` behaves differently and is also easy to misread. It returns a lazy iterator, so the exception surfaces only when iteration reaches the failed element, and everything after that element is abandoned.

In [ ]:
def explode(i):
    if i == 2:
        raise ValueError(f"task {i} failed")
    return i * 10


print("A. submit without inspecting the future")
with ThreadPoolExecutor(max_workers=4) as pool:
    for i in range(5):
        pool.submit(explode, i)          # the exception is stored and never read
print("   ...nothing was printed. The failure is gone.\n")

print("B. submit and inspect every future")
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(explode, i) for i in range(5)]
for f in futures:
    error = f.exception()
    print(f"   {'ok' if error is None else 'FAILED: ' + str(error)}")

print("\nC. pool.map stops at the first failure")
with ThreadPoolExecutor(max_workers=4) as pool:
    try:
        for value in pool.map(explode, range(5)):
            print(f"   got {value}")
    except ValueError as exc:
        print(f"   raised: {exc}  (results for 3 and 4 are never seen)")

The practical rule is that every submitted future must be inspected. If the result is not needed, call `exception()` and log anything that is not `None`, or attach a callback with `add_done_callback`. A `Future` whose outcome nobody reads is an error handler that nobody wrote.

## Step 3: Deadlines, and the limits of cancellation

A deadline over a group of futures is expressed by giving `as_completed` a timeout. When it expires, `as_completed` raises `TimeoutError` and you keep whatever arrived in time.

Then comes the uncomfortable part. `future.cancel()` succeeds only if the task is still sitting in the queue. Once a worker has picked it up, it cannot be cancelled, and there is no supported way to stop a running thread from outside. The thread will run to completion regardless, and the `with` block on the executor will wait for it, because `__exit__` calls `shutdown(wait=True)`.

The consequence is that a timeout in threaded Python bounds **how long you wait for an answer**, not how long the work runs. To bound the work itself, the work has to cooperate: check a shared `Event` at intervals, pass a deadline down to the socket layer as a connection and read timeout, or run the work in a separate process that can be terminated. In practice, setting timeouts on the underlying network client is the technique that matters most, because it addresses the actual cause.

In [ ]:
import threading
from concurrent.futures import TimeoutError as FuturesTimeout


def slow_service(name, seconds, cancelled):
    """A task that checks a cancellation flag between steps of its work."""
    steps = int(seconds / 0.05)
    for _ in range(steps):
        if cancelled.is_set():
            return {name: "cancelled early"}
        time.sleep(0.05)
    return {name: "complete"}


cancelled = threading.Event()
DEADLINE = 0.4

pool = ThreadPoolExecutor(max_workers=4)
futures = {
    pool.submit(slow_service, "fast", 0.1, cancelled): "fast",
    pool.submit(slow_service, "medium", 0.3, cancelled): "medium",
    pool.submit(slow_service, "glacial", 3.0, cancelled): "glacial",
}

screen = {}
start = time.perf_counter()
try:
    for future in as_completed(futures, timeout=DEADLINE):
        screen.update(future.result())
except FuturesTimeout:
    print(f"deadline of {DEADLINE}s reached after {time.perf_counter()-start:.2f}s")

print(f"sections ready : {sorted(screen)}")

# cancel() only works on tasks that have not started.
for future, name in futures.items():
    if not future.done():
        print(f"cancel({name}) returned {future.cancel()}  (running tasks cannot be cancelled)")

# The cooperative flag is what actually stops the work.
cancelled.set()
pool.shutdown(wait=True)
print(f"total elapsed  : {time.perf_counter()-start:.2f}s (not 3s, because the task checked the flag)")

## Step 4: The same orchestration in asyncio

asyncio handles this considerably better, because a task suspended at an `await` can be cancelled: the loop throws `CancelledError` into it at the suspension point, `finally` blocks run, and resources are released. Cancellation is genuine rather than advisory.

Three tools cover almost all orchestration:

- `asyncio.gather(*coros)` runs coroutines concurrently and returns results in argument order. By default the first exception propagates immediately while the other tasks keep running unattended, which is usually undesirable. `return_exceptions=True` collects failures as results instead.
- `asyncio.wait_for(coro, timeout)` bounds a single operation and cancels it on expiry. On Python 3.11 and later, `async with asyncio.timeout(seconds):` applies a deadline to an entire block and reads better for multi-step operations.
- `asyncio.TaskGroup`, added in 3.11, provides **structured concurrency**: tasks are created inside an `async with` block, and the block does not exit until all of them finish. If any task fails, the remaining ones are cancelled and the errors are raised together as an `ExceptionGroup`. No task can outlive the block, which removes an entire category of leaked background tasks.

The version below uses `gather` so it runs on any supported Python, and uses `TaskGroup` when the interpreter provides it.

In [ ]:
import asyncio
import sys

from clab import run_async


async def call_service_async(name, latency, fail=False):
    try:
        await asyncio.sleep(latency)
    except asyncio.CancelledError:
        # Cancellation arrives here as an exception. Clean up, then re-raise.
        print(f"   {name} was cancelled cleanly at its await point")
        raise
    if fail:
        raise ServiceError(f"{name} is unavailable")
    return {name: f"data from {name}"}


async def dashboard_async(deadline=1.0):
    async def guarded(name, latency, fail):
        try:
            return await call_service_async(name, latency, fail)
        except ServiceError as exc:
            return {"__degraded__": f"{name}: {exc}"}

    results = await asyncio.wait_for(
        asyncio.gather(*(guarded(n, l, f) for n, l, f in SERVICES)),
        timeout=deadline,
    )
    screen, degraded = {}, []
    for item in results:
        if "__degraded__" in item:
            degraded.append(item["__degraded__"])
        else:
            screen.update(item)
    return screen, degraded


with timed("v3  asyncio fan-out with a deadline"):
    screen, degraded = run_async(dashboard_async())

print(f"\nsections rendered : {sorted(screen)}")
print(f"degraded sections : {degraded}")

# What a missed deadline looks like: pending tasks are genuinely cancelled.
print("\nnow with a deadline the services cannot meet:")
try:
    run_async(dashboard_async(deadline=0.1))
except asyncio.TimeoutError:
    print("   TimeoutError raised, and every pending task was cancelled")


if sys.version_info >= (3, 11):
    async def dashboard_taskgroup():
        screen = {}
        async with asyncio.TaskGroup() as tg:          # structured concurrency
            tasks = {
                name: tg.create_task(call_service_async(name, latency, False))
                for name, latency, _ in SERVICES
            }
        for name, task in tasks.items():
            screen.update(task.result())
        return screen

    with timed("v4  asyncio.TaskGroup (Python 3.11+)"):
        screen = run_async(dashboard_taskgroup())
    print(f"\nsections rendered : {sorted(screen)}")
else:
    print("\n(TaskGroup requires Python 3.11 or newer; gather is the portable equivalent.)")

## Step 5: The cardinal sin, measured

Every coroutine on a loop shares one thread. A coroutine that runs without awaiting keeps that thread to itself, and every other task on the loop stops for exactly as long.

This is the failure mode that turns an async service into something slower than its synchronous predecessor, and it is almost always caused by one of two things: a synchronous I/O library called from a coroutine, such as `requests` or a blocking database driver, or a CPU-heavy computation performed inline.

The next cell measures it directly. A heartbeat task wakes every 20 milliseconds and records how late it was. A well-behaved loop keeps the lag near zero; a blocked loop shows lag equal to the length of the block.

In [ ]:
async def heartbeat(ticks=40, interval=0.02):
    """Records how late each 20ms tick actually was, in milliseconds."""
    lags = []
    for _ in range(ticks):
        started = time.perf_counter()
        await asyncio.sleep(interval)
        lags.append((time.perf_counter() - started - interval) * 1000)
    return lags


async def blocking_worker():
    time.sleep(0.5)              # a synchronous call inside a coroutine


async def cooperative_worker():
    await asyncio.sleep(0.5)     # the same wait, expressed as a suspension


async def measure(worker):
    beat = asyncio.create_task(heartbeat())
    # create_task only schedules; this yield lets the heartbeat actually start
    # before the worker runs, so the measurement covers the worker's behaviour.
    await asyncio.sleep(0)
    await worker()
    return await beat


for label, worker in [("time.sleep inside a coroutine", blocking_worker),
                      ("await asyncio.sleep", cooperative_worker)]:
    lags = run_async(measure(worker))
    print(f"{label:<32} max heartbeat lag {max(lags):7.1f} ms")

Half a second of blocking produces roughly half a second of lag on every other task. In a web server that is every in-flight request delayed by 500 milliseconds because one of them called a synchronous library.

## Step 6: Bridging blocking code into an event loop

Blocking code cannot always be avoided. The library may have no async version, or the work may be genuinely CPU-bound. The solution is to move it off the loop's thread.

`asyncio.to_thread(func, *args)` runs a blocking function on a worker thread and returns an awaitable. The loop stays free, and the calling coroutine suspends until the thread finishes. This is the right tool for blocking **I/O**, since the GIL is released during the wait and the worker thread costs almost nothing.

`loop.run_in_executor(executor, func, *args)` is the general form and accepts any executor. Passing a `ProcessPoolExecutor` sends the work to another process, which is the right tool for **CPU-bound** work: a thread would hold the GIL and slow the loop's own thread down even while running elsewhere.

The rule that follows: blocking I/O goes to a thread, heavy computation goes to a process, and neither belongs on the loop.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

from clab import fetch_image, make_thumbnail


async def threaded_worker():
    # Blocking I/O, moved off the loop onto a worker thread.
    await asyncio.to_thread(time.sleep, 0.5)


lags = run_async(measure(threaded_worker))
print(f"{'asyncio.to_thread(time.sleep)':<32} max heartbeat lag {max(lags):7.1f} ms")


async def cpu_on_the_loop(image):
    make_thumbnail(image)                 # pure Python, holds the GIL


async def cpu_in_a_process(image, pool):
    loop = asyncio.get_running_loop()
    await loop.run_in_executor(pool, make_thumbnail, image)


image = fetch_image("https://images.example.com/photo_00.jpg")

lags = run_async(measure(lambda: cpu_on_the_loop(image)))
print(f"{'CPU work inline on the loop':<32} max heartbeat lag {max(lags):7.1f} ms")

process_pool = ProcessPoolExecutor(max_workers=2)
process_pool.submit(len, [0]).result()      # start the workers before measuring
lags = run_async(measure(lambda: cpu_in_a_process(image, process_pool)))
print(f"{'CPU work in a process pool':<32} max heartbeat lag {max(lags):7.1f} ms")
process_pool.shutdown()

The bridge runs in the other direction too. A thread that needs to hand work to a loop running on a different thread uses `asyncio.run_coroutine_threadsafe(coro, loop)`, which schedules the coroutine on that loop and returns a `concurrent.futures.Future` the calling thread can wait on. This is how a synchronous callback from a legacy library reaches async code. Note that `asyncio.Queue`, `asyncio.Lock`, and the other asyncio primitives are **not** thread-safe, so a plain `q.put_nowait()` from another thread is a bug; the correct call is `loop.call_soon_threadsafe(q.put_nowait, item)`.

## Takeaways

- `submit` returns a `Future`; `as_completed` yields futures in completion order and accepts a group timeout.
- Every future must be inspected. Unread futures swallow exceptions completely, and `pool.map` abandons results after the first failure.
- Threads cannot be cancelled. A timeout bounds how long you wait, not how long the work runs. Bound the work with client timeouts, cooperative flags, or a separate process.
- asyncio cancellation is real: `CancelledError` is raised at the suspension point, `finally` blocks run, and `TaskGroup` guarantees no task outlives its block.
- A blocking call inside a coroutine delays every other task by its full duration. Use `asyncio.to_thread` for blocking I/O and `run_in_executor` with a process pool for computation.

## Interview follow-ups

*Recommendations time out for half of all users. What do you change?*

Separate the failure from the latency budget. Give that call its own short timeout, well inside the endpoint deadline, so it can fail fast without consuming the whole budget. Add a circuit breaker so that after a run of failures the call is skipped entirely for a cooling-off period, which protects both the dashboard and the struggling downstream service. Serve a cached or default set of recommendations when the breaker is open, and emit a metric for the degraded path so the fallback does not become invisible.

*The dashboard now needs data from a service that only has a synchronous client. Does the endpoint have to become synchronous?*

No. Wrap the client call in `asyncio.to_thread` so it runs on a worker thread while the loop stays free. The cost is one thread per concurrent call to that service, so bound it with a semaphore or a dedicated executor with a fixed size. This is a common and acceptable transitional design.

*How would you decide between `gather(return_exceptions=True)` and `TaskGroup`?*

`gather(return_exceptions=True)` suits partial success, where each result is independently useful and the caller decides what to do about failures. That is exactly the dashboard's requirement. `TaskGroup` suits all-or-nothing work, where one failure makes the rest pointless and the correct response is to cancel everything and propagate. Prefer `TaskGroup` for anything transactional, because it also guarantees that no task is left running after the block exits.

---

<a id="framework"></a>
# Choosing a Model Under Pressure

The six case studies reduce to a short sequence of questions. Asked in order, they select the right tool for almost every problem you will meet.

**1. Is the work waiting or computing?**

Measure rather than guess, as in Case Study 1. Waiting means network calls, disk reads, database queries, and message brokers. Computing means parsing, encoding, resizing, and numerical work in Python. Mixed workloads split into phases and each phase is answered separately.

**2. If it is waiting, how many operations run at once?**

Up to a few hundred, and especially if the libraries in the path are synchronous, use `ThreadPoolExecutor`. Thousands or more, or an existing async stack, use asyncio with async-native clients. Both overlap the same waiting; the difference is cost per unit and whether your I/O libraries cooperate.

**3. If it is computing, is it Python bytecode or a C extension?**

Pure Python holds the GIL and needs `ProcessPoolExecutor`. NumPy, Pillow, `zlib`, `hashlib` over large buffers and most compiled libraries release the GIL, so threads parallelise them effectively. Check before assuming.

**4. Is state shared?**

If nothing is shared, you are finished and the problem is easy. If state is shared, identify the invariant, find every operation that participates in it, and put all of them under one lock at the granularity of the invariant.

**5. Is the constraint mutual exclusion, capacity, an announcement, or a predicate?**

One at a time is `Lock`, or `RLock` when locked methods call each other. At most N at a time is `Semaphore`, preferably `BoundedSemaphore`. Has it happened yet is `Event`. Wait until the state is suitable is `Condition`, and if the shape is producer and consumer then `queue.Queue` already implements it.

**6. Is the work a batch or a stream?**

A finite batch is an executor. A continuous stream is a queue with worker threads, because that is where backpressure, multi-stage pipelines, and independent shutdown live.

**7. What happens when something fails or takes too long?**

Every future must be inspected. Every wait must have a timeout. Every deadline must be enforced somewhere that can actually stop the work: a client timeout, a cooperative flag, a cancellable task, or a separate process.

### Anti-pattern checklist

Run down this list before shipping concurrent code.

| Symptom | Likely cause |
|---|---|
| Threads made it slower | CPU-bound work under the GIL |
| Works locally, fails in production | A race whose window is widened by real latency |
| Results occasionally wrong, never reproducible | Check-then-act without a lock |
| Service hangs with no error | Deadlock, a leaked semaphore permit, or a dead worker with an unfinished task |
| Memory grows until the process is killed | A queue without `maxsize` |
| Async service slower than the sync one it replaced | A blocking call inside a coroutine |
| Background failures never surface | A `Future` whose result nobody read |
| Multiprocessing fails only on macOS or Windows | Spawn cannot import a function defined in `__main__` or a notebook cell |
| Everything serialises despite fine-grained locks | A lock held across a blocking call |
| Retries make the outage worse | No jitter, so every client retries in the same instant |

---

<a id="summary"></a>
# Summary and Key Takeaways

Concurrency is the structural property of dividing work into independently schedulable units. Parallelism is the execution property of running units simultaneously on several processors. A workload is I/O-bound when elapsed time is dominated by waiting and CPU-bound when it is dominated by computation, and that classification determines the model before any other consideration.

CPython's Global Interpreter Lock permits one thread at a time to execute Python bytecode. Threads therefore overlap waiting but not computing, and only separate processes give parallel Python computation. The GIL makes a single bytecode instruction atomic and gives no guarantee about a statement, a method, or any longer sequence, which is why shared state still needs synchronisation.

A race condition is a defect whose outcome depends on timing that is not controlled. Its most common shape is check-then-act, and its usual symptom is a lost update. A lock makes a sequence atomic and must be sized to the invariant rather than to a variable: per SKU, per key, per account. Fine-grained locks improve throughput and increase the risk of a cycle in the wait graph, which a global lock ordering removes.

Semaphores bound how many operations run at once and belong to the resource rather than to any one pool. Token buckets bound how many start per unit of time. Events announce that something has happened, broadcast to every waiter and remain set for late arrivals. Conditions let a thread wait for an arbitrary predicate and must always be used with a `while` loop that re-tests it. `queue.Queue` combines a lock and two conditions into a correct bounded buffer, and its `maxsize` is what converts a rate mismatch into backpressure instead of a memory leak.

Executors are the right default for finite batches, and every `Future` they return must be inspected or its exception is lost. Threads cannot be cancelled, so timeouts bound waiting rather than work, and bounding work requires client timeouts, cooperative flags, or a separate process. asyncio replaces preemption with cooperation: a coroutine can only be interrupted at an `await`, which eliminates most races, makes cancellation genuine, and imposes one absolute rule, that nothing may block the loop. Blocking I/O goes to a thread with `asyncio.to_thread`, and computation goes to a process with `run_in_executor`.

### Key takeaways

1. Measure the split between waiting and computing before choosing a model.
2. Threads overlap waiting. Processes overlap computing. Coroutines overlap waiting at a far lower cost per unit.
3. The GIL protects one bytecode instruction and nothing larger.
4. Locks protect invariants. Find every operation that participates in the invariant.
5. Match lock granularity to the invariant, and acquire multiple locks in a fixed global order.
6. Semaphores enforce concurrency limits; token buckets enforce rate limits; the two are different constraints.
7. Queues give backpressure, and `maxsize` is what makes overload survivable.
8. Call `task_done()` in a `finally` block and catch exceptions inside worker loops.
9. Inspect every future, and give every wait a timeout.
10. Never block the event loop, and never hold a lock across a blocking call.

---

<a id="practice"></a>
# Practice Problems

Work through these before reading the solutions. Problems 1 to 4 are diagnostic, in the style of a code review round. Problems 5 to 8 are design questions, in the style of a system design round.

### Problem 1: Diagnose the counter

```python
class MetricsCollector:
    def __init__(self):
        self.counts = {}
        self._lock = threading.Lock()

    def record(self, metric):
        if metric not in self.counts:
            with self._lock:
                self.counts[metric] = 0
        with self._lock:
            self.counts[metric] += 1
```

Two threads call `record("requests")` concurrently on a fresh collector. Describe precisely what can go wrong, and fix it.

### Problem 2: Why is this slower?

A developer replaces a sequential loop that hashes 200 files with a `ThreadPoolExecutor(max_workers=8)` and reports that the threaded version is 15 percent slower. Hashing is done with a pure-Python implementation. Explain the result and state what you would change.

### Problem 3: The pipeline that stops

```python
q = queue.Queue()

def worker():
    while True:
        item = q.get()
        result = transform(item)
        db.write(result)
        q.task_done()

for _ in range(4):
    threading.Thread(target=worker, daemon=True).start()

for item in items:
    q.put(item)
q.join()
```

After several hours in production the process stops making progress. No exception is logged and the process is still running. Give two independent explanations and the corresponding fixes.

### Problem 4: The async endpoint that got slower

A team migrates an endpoint from Flask with a thread pool to FastAPI with asyncio. Throughput drops by a factor of five. The endpoint reads a row with `psycopg2`, calls one internal service with `requests`, and returns JSON. Explain the drop and give two possible fixes with their trade-offs.

### Problem 5: Design a bounded worker pool with retries

Design a component that accepts jobs, runs at most N concurrently, retries a failed job up to three times with exponential backoff, sends permanently failed jobs to a dead-letter queue, and can be shut down without losing accepted work. Specify the primitives you would use and why.

### Problem 6: Design a concurrent web crawler

Design a crawler that fetches up to 10,000 pages, never visits a URL twice, respects a limit of 5 concurrent requests per host, and stays within 500 MB of memory. State the concurrency model and each synchronisation primitive you would use.

### Problem 7: The seat reservation service

A ticketing service holds a seat for ten minutes while the buyer pays. Requests arrive on many threads across four application servers. Design the reservation operation. Explain what breaks if the design uses only a `threading.Lock`.

### Problem 8: Explain the measurement

A batch job downloads 500 files. With `ThreadPoolExecutor(max_workers=10)` it takes 50 seconds. With `max_workers=100` it takes 48 seconds. With `max_workers=500` it takes 71 seconds. Explain the shape of that curve and describe how you would pick the worker count.

---

## Solutions

### Solution 1

The method contains a check-then-act race, and the lock is placed so that it protects neither half properly. Thread A evaluates `metric not in self.counts` and finds it absent. Thread B evaluates the same expression and also finds it absent. Both acquire the lock in turn and both execute `self.counts[metric] = 0`. If A had already incremented the count to 1 between the two blocks, B's initialisation resets it to 0 and A's increment is lost.

The defect is that the check sits outside the lock while the act sits inside it, so the two are not atomic together. The fix is to place the entire sequence in one critical section:

```python
def record(self, metric):
    with self._lock:
        self.counts[metric] = self.counts.get(metric, 0) + 1
```

A cleaner alternative removes the branch entirely with `collections.defaultdict(int)`, though the lock is still required because `counts[metric] += 1` is a read, an add, and a write. A third option is `collections.Counter` under the same lock. The general lesson is that acquiring a lock twice in a row is not the same as holding it once across both operations.

### Solution 2

Hashing implemented in pure Python executes as Python bytecode, so a thread performing it holds the GIL for the entire computation. Eight threads therefore take turns rather than running in parallel, and the total computation time is unchanged. The 15 percent regression is the cost of the change: thread creation, task dispatch, and the interpreter switching between threads every few milliseconds, each switch involving GIL handoff and cache disturbance.

The fix is `ProcessPoolExecutor` with `max_workers=os.cpu_count()` and a `chunksize` large enough that dispatch cost is small relative to per-item work. A better fix in this specific case is to stop hashing in pure Python: `hashlib` releases the GIL for large buffers, so switching to `hashlib` would make the original thread pool effective and avoid the serialisation cost of processes entirely. This is the general point that the GIL blocks Python bytecode, so the first question about any CPU-bound hot path is how much of it actually runs as bytecode.

### Solution 3

**Explanation one: a worker died.** If `transform` or `db.write` raises, the exception propagates out of `worker`, the thread ends, and `task_done()` is never called for the item it held. Each such failure permanently removes one worker, and after four failures the pipeline has no workers at all while `q.join()` waits forever on unfinished tasks that will never be marked done. The fix is to wrap the body in `try`/`except` so a failing item is logged and counted rather than fatal, and to move `task_done()` into a `finally` block so the bookkeeping is correct in every path.

**Explanation two: the queue is unbounded and a downstream dependency has slowed.** If `db.write` becomes slower than the arrival rate, the queue grows without limit. The process may be swapping, or close to an out-of-memory kill, and appears stalled. The fix is `queue.Queue(maxsize=N)`, which blocks the producer once the backlog reaches N and pushes the slowdown upstream where it can be observed and handled.

A third possibility worth mentioning in an interview is that `db.write` is blocked on a connection with no timeout, so the workers are alive but permanently waiting. The fix is a timeout on every network call, plus retries with backoff.

### Solution 4

Both `psycopg2` and `requests` are synchronous. Called from a coroutine, each one blocks the event loop thread for the full duration of its network round trip, and because a single loop serves every request, all concurrent requests are serialised behind it. The Flask version had a pool of threads, so N requests waited in parallel; the asyncio version has one thread, so N requests wait one after another. The migration removed the concurrency without replacing it.

**Fix one: use async-native clients**, `asyncpg` or `psycopg` version 3 in async mode, and `httpx` or `aiohttp`. This is the correct destination and gives the full benefit of the model. The cost is a rewrite of the data access layer and any library that depends on it.

**Fix two: wrap the blocking calls in `asyncio.to_thread`.** This restores concurrency immediately with a very small change and keeps the loop free. The cost is one worker thread per in-flight call, which reintroduces the memory and scheduling cost of threads for those calls and caps concurrency at the executor size, so the ceiling is lower than a fully async stack. It is a sound transitional step and is often the right first move.

A third answer worth stating: if nothing in the endpoint is going to become async-native, running the sync framework under a thread-per-request server was the correct architecture, and reverting is legitimate.

### Solution 5

**Structure.** A `queue.Queue(maxsize=M)` holds accepted jobs and provides backpressure, so `submit` blocks when the backlog is full instead of growing without limit. A fixed pool of N worker threads consumes it, which bounds concurrency by construction; if the concurrency limit is shared with other components, use a `BoundedSemaphore` instead so the limit lives with the resource.

**Retries.** Retry inside the worker rather than by re-queuing, so a job's attempt count travels with it. Wrap each attempt in `try`/`except`, sleep for `base * 2 ** attempt` plus random jitter between attempts, and give up after three. Jitter is essential: without it, a downstream outage causes every retry to land in the same instant and reproduces the load spike that caused the failure.

**Dead letters.** A permanently failed job goes to a second queue or a durable store, with the exception, the attempt count and the payload, so it can be inspected and replayed.

**Shutdown.** A `threading.Event` signals shutdown, and workers use `q.get(timeout=...)` so they can observe the flag while idle. Graceful shutdown stops accepting new work, calls `q.join()` to drain the backlog, sets the event, and joins the workers with a timeout. Retry sleeps should wait on the event rather than calling `time.sleep`, so a shutdown during a backoff interval is not delayed by the full backoff.

**Correctness details.** `task_done()` in a `finally` block, every exception caught inside the worker loop, and a bounded total time per job so one pathological item cannot occupy a worker indefinitely.

### Solution 6

**Model.** The workload is almost entirely network waiting, and the target of 10,000 pages implies high concurrency, so asyncio with `aiohttp` or `httpx` is the natural choice. A thread pool would need hundreds of threads to reach the same throughput and would cost far more memory.

**Frontier.** An `asyncio.Queue` with a `maxsize` holds URLs to visit. The bound is what keeps memory flat: without it, a crawl of a densely linked site expands the frontier faster than workers drain it.

**Deduplication.** A `set` of normalised URLs, consulted and updated before enqueuing. Because all coroutines run on one loop and the check-and-add contains no `await`, it is atomic without a lock. Normalisation matters more than the data structure: lowercase the host, strip fragments, sort query parameters, and resolve relative links, or the same page is crawled under many spellings. At 10,000 URLs a set is fine; at 100 million it becomes a Bloom filter plus a durable store.

**Per-host limit.** A dictionary mapping host to `asyncio.Semaphore(5)`, created on first use, acquired around each fetch. This is the per-key pattern from Case Study 5, applied to a capacity constraint rather than a load.

**Memory.** Bound the frontier with `maxsize`, cap the response body size read per page, parse and discard rather than accumulating documents, and write extracted results to disk or a database incrementally.

**Robustness.** A per-request timeout, retries with backoff and jitter for transient failures, a depth or page limit as a stopping condition, and `robots.txt` plus a politeness delay per host.

### Solution 7

**The operation.** Reserving a seat is a check-then-act over shared state that must remain atomic: confirm the seat is free, mark it held, and record an expiry ten minutes ahead. The three steps have to be one atomic unit, and the state must be visible to all four servers.

**Why a `threading.Lock` fails.** It is an object in one process's memory. The other three servers have their own copies, see nothing, and are not excluded by it. Four servers can each admit one buyer for the same seat, and the lock provides no protection whatever across the fleet. This is the single most common wrong answer to this question in interviews.

**What to use instead.** Push the atomicity into the shared store. A conditional update is the simplest correct approach:

```sql
UPDATE seats
   SET status = 'held', held_by = ?, hold_expires_at = now() + interval '10 minutes'
 WHERE seat_id = ?
   AND (status = 'free' OR (status = 'held' AND hold_expires_at < now()))
```

The database evaluates the condition and performs the write as one atomic statement, and the affected row count tells the caller whether it won. Expired holds are reclaimed by the same statement, so nothing depends on a cleanup job running promptly. A background sweeper still helps for reporting and for releasing inventory that would otherwise look held.

The alternative is an atomic operation in Redis, such as `SET seat:{id} {buyer} NX PX 600000`, which is faster and gives a shorter durability guarantee. If the payment record and the seat record must move together, prefer the database so both changes sit in one transaction.

**Confirmation and expiry.** Confirming payment is a second conditional update that requires the hold to still belong to this buyer and to be unexpired, which prevents a late payment from claiming a seat that has already been re-sold. The whole design is idempotent if the client sends a reservation id, so a retried request after a timeout does not create a second hold.

### Solution 8

The curve has three regions, and each has a different cause.

**Ten to one hundred workers: saturation.** At ten workers the job is limited by concurrency, so adding workers helps. Somewhere below one hundred the bottleneck moves elsewhere, most likely to available bandwidth or to a limit on the server side, and additional workers no longer reduce elapsed time. The two-second gain between ten and one hundred shows the system was already close to saturated at ten.

**One hundred to five hundred workers: contention.** Beyond saturation, more workers add cost without adding throughput: memory for stacks, scheduler pressure, more connections than the remote server wants to hold open, and possibly server-side throttling or queueing that makes each individual request slower. The 71 seconds is the original work plus the overhead of managing five hundred threads that spend most of their time competing.

**How to choose.** Measure rather than reason. Sweep the worker count over a range, plot elapsed time and error rate, and choose a value on the flat part of the curve just past the knee, biased low. A low value on the plateau gives nearly the best time with the least memory, the fewest connections and the most headroom for the remote service. Two constraints override the measurement: any concurrency limit imposed by the provider is a correctness bound rather than a tuning parameter, and the error rate matters more than elapsed time, since a configuration that is two seconds faster and produces rejections is worse than one that is slower and clean.

---

<a id="whats-next"></a>
## Where to Go Next

Three directions extend this material.

**Correctness tooling.** Race conditions are hard to test for and easy to reason about incorrectly. `sys.setswitchinterval` makes thread switches far more frequent and turns rare races into reproducible ones. Deterministic tests assert on invariants and on call counts rather than on timings. Property-based testing over operation interleavings catches cases a hand-written test will not.

**Distributed concurrency.** Every primitive in this workbook lives inside a single process. Across machines, shared memory disappears and communication becomes unreliable, so locks give way to conditional writes, message queues, idempotency keys, leases with fencing tokens, and consensus protocols. Problem 7 is the smallest complete example of that transition.

**Free-threaded Python.** PEP 703 introduced a build of CPython without the GIL, available experimentally from 3.13. In that build, threads do execute Python bytecode in parallel, which changes the answer to "how do I parallelise computation" and changes nothing about correctness. Every lock, every invariant and every race in this workbook remains exactly as relevant, and several become easier to trigger, because the accidental protection that the GIL provided for short sequences is gone.

---

<sub>**Codeverra** · A learning platform to master coding, data science, DSA, and AI.<br>
Learn more at [codeverra.com](https://codeverra.com)</sub>